## Step 3A: Generate new molecules and reaction pathways with DORAnet

This step expands each starter molecule into an enzymatic reaction network with
`DORAnet`, enumerates the molecules reachable from that starter, and finds
biosynthetic pathways to each of them. It runs from the command line.

### Directory layout

All generation runs for this target live under `Step3A_Antibiotic_Compounds/`. It
holds two sources that differ in how the starters are organized on disk:

```
Step3A_Antibiotic_Compounds/
├── Step2_BasidalinPrecursor/     # many precursor starters, run in parallel
│   ├── runDORAnet_parallel.py
│   ├── doranet_config_parallel.yaml
│   └── doranet_output/
│       ├── starter_00000/
│       ├── starter_00001/
│       └── ...
└── Step2_Basidalin/              # one flat run for the basidalin scaffold itself
    ├── runDORAnet.py
    ├── doranet_config.yaml
    └── ...
```

`Step2_BasidalinPrecursor` fans a whole CSV of precursor SMILES out into one
DORAnet job per starter. `runDORAnet_parallel.py` reads
`doranet_config_parallel.yaml`, and for every SMILES it creates
`doranet_output/starter_XXXXX/`, writes a self-contained `runDORAnet.py` and
`doranet_config.yaml` into it, and then runs `python runDORAnet.py doranet_config.yaml`
there. Each `starter_*` directory is therefore a complete, independently runnable
DORAnet job.

`Step2_Basidalin` is a single flat run for the basidalin scaffold itself (the
`BasidalinOnly_gen3` job), with its `runDORAnet.py` and `doranet_config.yaml`
directly in the directory, read the same way as the Inosine starter.

Each `doranet_config.yaml` sets the DORAnet checkout path (`doranetPath`), the
output prefix (`jobName`), the `starters` SMILES, the network settings
(`generations`, `direction`, `ruleset`, `maxAtoms`), the pathway depth
(`totalGenerations`), and an optional `helpers` set. For the precursor source you
do not edit these per-directory files by hand; they are generated from
`doranet_config_parallel.yaml`, so you change the network settings there once and
re-scaffold.

### How to run

Precursor source (many starters). First scaffold and run every starter in parallel:

```bash
cd Step3A_Antibiotic_Compounds/Step2_BasidalinPrecursor
python runDORAnet_parallel.py doranet_config_parallel.yaml
```

Set `execution.run: true` in `doranet_config_parallel.yaml` to have the
orchestrator run the jobs across `parallel.num_workers`. Set `execution.run: false`
to only create the `starter_*` directories and their files, then run them yourself,
one at a time or on the HPC scheduler:

```bash
for d in doranet_output/starter_*/ ; do
    ( cd "$d" && python runDORAnet.py doranet_config.yaml )
done
```

Basidalin scaffold source (single starter):

```bash
cd Step3A_Antibiotic_Compounds/Step2_Basidalin
python runDORAnet.py doranet_config.yaml
```

For large networks, submit each starter to the HPC scheduler instead of running
interactively, since generation and pathway search can be memory- and
time-intensive. Scaffolding with `execution.run: false` first is the usual way to
prepare a large precursor batch for scheduler submission.

### Prerequisites

- A local DORAnet checkout at the `doranetPath` set in each config (or
  `doranet.path` in the parallel config); the runner adds it to `sys.path` at runtime.
- An active environment that includes DORAnet, RDKit, and PyYAML
  (`SynThera_ExptDesigns`).
- Runtime scales with network size: every generated molecule except the starters
  and helpers becomes a pathway target, so tune `generations` and `maxAtoms` to
  keep runs tractable.

### Outputs and next step

Each `starter_*` directory (precursor) and the `Step2_Basidalin` directory hold
their own outputs, prefixed with that run's `jobName`:

- `{jobName}_molecules.csv` — every generated molecule with its formula, exact
  mass, and heavy-atom count, flagged by whether it is a starter.
- `{jobName}_network_pretreated.json` and the pathway files with the same
  `{jobName}` prefix, describing the biosynthetic routes to each target.

These generated molecules feed the next stage (Step 3B in this notebook), where
novel drug-like compounds are filtered from the retro-biosynthesis output across
both sources and then scored for potency and ADMET properties.

# Step 3B: Aggregate DORAnet-generated antibiotics across both Basidalin sources

This notebook post-processes the DORAnet runs under
`Step3A_Antibiotic_Compounds/`. It reads two sources that have different on-disk
layouts and unifies them through a single per-starter processor:

- `Step2_BasidalinPrecursor/doranet_output/starter_*` — many starter
  subdirectories, exactly the layout used by `generatedMolecules.ipynb`. Each
  `starter_*` folder is treated as one starter.
- `Step2_Basidalin` — a single flat starter directory, read the same way the
  `Inosine` starter was (its `*_uniqueMolecules.csv` plus its network file).

For every starter it traces which molecules are derived from that starter,
filters out cofactors and non-drug-like products, and separates the
high-feasibility subset using the DORA-XGB (rule3) reaction scores when a
feasibility table is available. Per-starter files are written to `resultsDir`,
then aggregated into two master tables: all possible target compounds, and the
high-feasibility target compounds.


In [27]:
import os
import io
import multiprocessing
from contextlib import redirect_stdout
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import re
import math
import json
import glob
import yaml
import base64
from io import BytesIO
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, Draw, QED, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.SimDivFilters.rdSimDivPickers import MaxMinPicker

from PIL import Image, ImageDraw, ImageFont
from IPython.display import HTML, display, SVG

RDLogger.DisableLog('rdApp.*')

from DORA_XGB import DORA_XGB
by_desc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_descending_MW')
by_asc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_ascending_MW')
add_concat_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_concat')
add_subtract_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_subtract')

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol, canonical=True) if mol else None

## Configuration

In [28]:
# Anchor the project root explicitly so paths do not depend on the launch directory.
projectRoot = os.path.expanduser("~/DTRA_project/MACAW/SynThera")

# Set target to the pathogen present in your data.
target = "BacillusAnthracis"

# Results location, derived from the project root. All outputs land here.
resultsDir = os.path.join(projectRoot, target, f"Results_{target}")
os.makedirs(resultsDir, exist_ok=True)

# Parent directory that holds this target's generated-compound sources.
compoundsRootDir = os.path.join(resultsDir, "Step3A_Antibiotic_Compounds")

# Two DORAnet sources with different on-disk layouts.
# Step2_BasidalinPrecursor keeps many starter_* subdirectories under doranet_output,
# exactly like the generatedMolecules pipeline: each starter_* is one starter with its
# own starter_N_molecules.csv and starter_N_network_pretreated.json.
# Step2_Basidalin is one flat single-starter run (the basidalin scaffold itself), read
# like the Inosine starter, but its files use the BasidalinOnly_gen3 prefix and it ships
# no precomputed feasibility table.
BasidalinprecursorOutputDir = os.path.join(compoundsRootDir, "Step2_BasidalinPrecursor", "doranet_output")
basidalinStarterDir = os.path.join(compoundsRootDir, "Step2_Basidalin")
basidalinStarterName = "Basidalin"

# The single true starter for Step2_Basidalin. Given explicitly because that run's
# uniqueMolecules.csv marks reactant-only intermediates as Is_Starter too, so the column
# cannot be trusted to recover the real starter.
basidalinStarterSmiles = r"C1=C(/C(=C\C=O)/OC1=O)N"

# DORAnet cofactor table (used to exclude cofactors and endogenous metabolites).
doranetCofactorsPath = os.path.join(projectRoot, "Input_Data", "doranet_all_cofactors.tsv")

# Helper / small molecules to exclude from the derived compound set.
helpersToExclude = {
    "O", "O=O", "[H][H]", "O=C=O", "C=O", "[C-]#[O+]", "Br", "[Br][Br]",
    "CO", "C=C", "O=S(O)O", "N", "O=S(=O)(O)O", "O=NO", "N#N",
    "O=[N+]([O-])O", "NO", "C#N", "S", "O=S=O", "N#CO", "[H+]", "OO",
    "Cl", "I", "O=C(O)O", "O=P(O)(O)O", "O=P(O)(O)OP(=O)(O)O", "C",
    "CC", "CC=O", "CC(=O)O", "CCC(=O)O",
}

# Fingerprint, drug-likeness, and cofactor/endogenous cutoffs.
fpRadius, fpBits = 2, 2048
lipinskiMaxViolations = 0
physicochemicalCutoffs = {"mwMax": 700.0, "tpsaMax": 200.0, "hbdMax": 7, "qedMin": 0.10}

# DORA-XGB feasibility settings. rule3 (add_concat) is the published model and is applied
# uniformly across both sources, including live scoring of the Basidalin run.
feasibilityLabelCol = "feasibilityLabel_rule3"
feasibilityScoreCol = "feasibilityScore_rule3"
feasibilityTargetValue = 1
feasibilityFileName = "reactionDF_wDORAXGBfeasibility.csv"

# Fallback network depth if a starter doranet_config.yaml lacks 'generations'.
defaultGenerations = 3

# Tables used only to label pathway figures (optional; degrade gracefully if absent).
cofactorTablePathForPlots = doranetCofactorsPath
rulesetPathForPlots = os.path.join(projectRoot, "Input_Data", "JN3604IMT_rules.tsv")

# Load the DORAnet cofactor set once (starter-independent).
_cofactorTable = pd.read_csv(doranetCofactorsPath, sep="\t")
_cofactorTable.columns = [col.lstrip("#").strip() for col in _cofactorTable.columns]
cofactorExclusionSet = {
    Chem.MolToSmiles(mol)
    for smi in _cofactorTable["SMILES"].dropna()
    if (mol := Chem.MolFromSmiles(str(smi))) is not None
}

print("Target:", target)
print("Compounds root:", os.path.abspath(compoundsRootDir))
print("Precursor doranet_output:", os.path.abspath(BasidalinprecursorOutputDir))
print("Basidalin starter dir:", os.path.abspath(basidalinStarterDir))
print("Results directory:", os.path.abspath(resultsDir))
print("Cofactors loaded:", len(cofactorExclusionSet))


Target: BacillusAnthracis
Compounds root: /users/sghosh6/DTRA_project/MACAW/SynThera/BacillusAnthracis/Results_BacillusAnthracis/Step3A_Antibiotic_Compounds
Precursor doranet_output: /users/sghosh6/DTRA_project/MACAW/SynThera/BacillusAnthracis/Results_BacillusAnthracis/Step3A_Antibiotic_Compounds/Step2_BasidalinPrecursor/doranet_output
Basidalin starter dir: /users/sghosh6/DTRA_project/MACAW/SynThera/BacillusAnthracis/Results_BacillusAnthracis/Step3A_Antibiotic_Compounds/Step2_Basidalin
Results directory: /users/sghosh6/DTRA_project/MACAW/SynThera/BacillusAnthracis/Results_BacillusAnthracis
Cofactors loaded: 43


### Helper functions

In [29]:
def splitMoleculeString(moleculeString):
    if moleculeString is None or (isinstance(moleculeString, float) and pd.isna(moleculeString)):
        return []
    return [mol for mol in str(moleculeString).split(".") if mol]


def readReactionDf(jsonPath):
    """Parse a DORAnet *_network_pretreated.json into a reaction DataFrame."""
    with open(jsonPath, "r", encoding="utf-8") as handle:
        reactionList = json.load(handle)

    records = []
    for rxn in reactionList:
        parts = str(rxn).split(">")
        if len(parts) != 4:
            continue
        reactants, ruleName, metaBlock, products = parts
        metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
        thermo, reactantStoich, productStoich, reactionType = metaParts
        records.append({
            "reactants": reactants.strip(),
            "products": products.strip(),
            "reactionString": f"{reactants.strip()} >> {products.strip()}",
            "ruleName": ruleName.strip(),
            "thermo": thermo,
            "reactionType": reactionType,
        })
    return pd.DataFrame(records)


def morganFp(mol):
    return rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, fpRadius, nBits=fpBits) if mol is not None else None


def isCofactorOrEndogenous(mol, canonSmi, cofactorExclusionSet, cutoffs):
    if mol is None:
        return True
    if canonSmi in cofactorExclusionSet:
        return True
    if (Descriptors.MolWt(mol) > cutoffs["mwMax"]
            or rdMolDescriptors.CalcTPSA(mol) > cutoffs["tpsaMax"]
            or rdMolDescriptors.CalcNumHBD(mol) > cutoffs["hbdMax"]):
        return True
    if QED.qed(mol) < cutoffs["qedMin"]:
        return True
    return False


def lipinskiViolationCount(mol):
    if mol is None:
        return None
    violations = 0
    if Descriptors.MolWt(mol) > 500:
        violations += 1
    if Descriptors.MolLogP(mol) > 5:
        violations += 1
    if rdMolDescriptors.CalcNumHBD(mol) > 5:
        violations += 1
    if rdMolDescriptors.CalcNumHBA(mol) > 10:
        violations += 1
    return violations


def passesLipinski(canonSmi, maxViolations):
    violations = lipinskiViolationCount(Chem.MolFromSmiles(str(canonSmi)))
    return violations is not None and violations <= maxViolations


def nearestStarterBySimilarity(mol, starterFpByCanon):
    productFp = morganFp(mol)
    if productFp is None or not starterFpByCanon:
        return None, -1.0
    bestStarter, bestSimilarity = None, -1.0
    for starterCanon, starterFp in starterFpByCanon.items():
        similarity = DataStructs.TanimotoSimilarity(productFp, starterFp)
        if similarity > bestSimilarity:
            bestSimilarity, bestStarter = similarity, starterCanon
    return bestStarter, bestSimilarity


def buildStarterDerivedSet(reactionDF, starterMols, canonStarterSet, canonHelperSet,
                           cofactorExclusionSet, cutoffs, starterFpByCanon):
    molToReactionIdxs = defaultdict(list)
    for idx, row in reactionDF.iterrows():
        for smi in splitMoleculeString(row["reactants"]):
            molToReactionIdxs[canonicalizeSmiles(smi)].append(idx)

    starterDerivedSet = {canonSmi: (canonSmi, 0, 1.0, None) for canonSmi, _ in starterMols}
    queue = deque((canonSmi, 0) for canonSmi in canonStarterSet)
    visited = set(canonStarterSet)

    while queue:
        currentCanon, currentDepth = queue.popleft()
        if currentCanon in canonHelperSet:
            continue

        for rxnIdx in molToReactionIdxs.get(currentCanon, []):
            row = reactionDF.loc[rxnIdx]
            productScores = []
            for productSmi in splitMoleculeString(row["products"]):
                productCanon = canonicalizeSmiles(productSmi)
                if productCanon in canonHelperSet or productCanon in starterDerivedSet:
                    continue
                productMol = Chem.MolFromSmiles(str(productSmi))
                if isCofactorOrEndogenous(productMol, productCanon, cofactorExclusionSet, cutoffs):
                    continue
                matchedStarter, similarity = nearestStarterBySimilarity(productMol, starterFpByCanon)
                if matchedStarter is None:
                    continue
                productScores.append((similarity, productCanon, matchedStarter))

            if not productScores:
                continue

            productScores.sort(key=lambda t: t[0], reverse=True)
            keptSimilarity, bestProductCanon, bestMatchedStarter = productScores[0]
            runnerUpSimilarity = productScores[1][0] if len(productScores) > 1 else None

            if bestProductCanon not in visited:
                visited.add(bestProductCanon)
                starterDerivedSet[bestProductCanon] = (
                    bestMatchedStarter, currentDepth + 1, keptSimilarity, runnerUpSimilarity
                )
                queue.append((bestProductCanon, currentDepth + 1))

    return starterDerivedSet


def enumerateStarterDerivedProducts(uniqueProductList, starterDerivedSet, canonHelperSet,
                                    canonStarterSet, cofactorExclusionSet, cutoffs):
    records = []
    for smi in uniqueProductList:
        mol = Chem.MolFromSmiles(str(smi))
        canonSmi = Chem.MolToSmiles(mol) if mol else None
        if canonSmi is None or canonSmi in canonHelperSet or canonSmi in canonStarterSet:
            continue
        if isCofactorOrEndogenous(mol, canonSmi, cofactorExclusionSet, cutoffs):
            continue
        if canonSmi not in starterDerivedSet:
            continue
        matchedRaw, generationDepth, keptSim, runnerUp = starterDerivedSet.get(
            canonSmi, (None, None, None, None)
        )
        records.append({
            "starter_Canonical_SMILES": canonicalizeSmiles(matchedRaw) if matchedRaw else None,
            "Canonical_SMILES": canonSmi,
            "generationSteps": generationDepth,
            "DerivativeSimScore": round(keptSim, 4) if keptSim is not None else None,
            "NonDerivativeSimScore": round(runnerUp, 4) if runnerUp is not None else None,
            "_molecularWeight": round(Descriptors.MolWt(mol), 4),
        })
    return pd.DataFrame(records)


def ensureReactantProductCols(df):
    df = df.copy()
    if "reactants" in df.columns and "products" in df.columns:
        return df.reset_index(drop=True)
    if "reactionString" in df.columns:
        sides = df["reactionString"].astype(str).str.split(">>", n=1, expand=True)
        df["reactants"] = sides[0].str.replace(" ", "", regex=False)
        df["products"] = sides[1].str.replace(" ", "", regex=False)
        return df.reset_index(drop=True)
    raise KeyError("Feasibility table needs 'reactants'/'products' or 'reactionString' columns")


def dedupLipinskiSort(rawDF):
    """Deduplicate on canonical SMILES, keep Lipinski-compliant rows, sort by MW."""
    if rawDF.empty:
        return rawDF
    dedup = rawDF.drop_duplicates(subset="Canonical_SMILES", keep="first").reset_index(drop=True)
    mask = dedup["Canonical_SMILES"].apply(lambda s: passesLipinski(s, lipinskiMaxViolations))
    kept = dedup[mask].reset_index(drop=True)
    return (
        kept.sort_values("_molecularWeight", ascending=False)
        .drop(columns="_molecularWeight")
        .reset_index(drop=True)
    )


def buildFeasibilityScoreByCompound(reactionDfHighFeasible, scoreCol):
    """Assign each compound the maximum rule3 score among high-feasibility
    reactions that produce it."""
    scoreByCompound = {}
    if scoreCol not in reactionDfHighFeasible.columns:
        return scoreByCompound
    for _, row in reactionDfHighFeasible.iterrows():
        try:
            score = float(row[scoreCol])
        except (TypeError, ValueError):
            continue
        for productSmi in splitMoleculeString(row["products"]):
            canon = canonicalizeSmiles(productSmi)
            if canon is None:
                continue
            if canon not in scoreByCompound or score > scoreByCompound[canon]:
                scoreByCompound[canon] = score
    return scoreByCompound


def readGenerationsFromConfig(starterDir, default):
    """Read the DORAnet network depth (generations) from the starter's config."""
    configPath = os.path.join(starterDir, "doranet_config.yaml")
    if not os.path.isfile(configPath):
        print(f"[warn] {os.path.basename(starterDir)}: no doranet_config.yaml; using generations={default}")
        return default
    with open(configPath) as handle:
        cfg = yaml.safe_load(handle) or {}
    return int(cfg.get("generations", default))

# DORA-XGB model per rule, so live scoring stays consistent with feasibilityLabelCol.
feasibilityModelByRule = {
    "rule1": by_desc_MW_model,
    "rule2": by_asc_MW_model,
    "rule3": add_concat_model,
    "rule4": add_subtract_model,
}


def computeFeasibilityLive(reactionDF):
    """Score a reaction network in place with DORA-XGB when no precomputed feasibility
    table exists (the Basidalin single-starter run). Uses the model that matches
    feasibilityLabelCol so the rule stays consistent with the configuration."""
    if reactionDF.empty:
        return reactionDF.copy()
    rule = feasibilityLabelCol.split("_")[-1]
    model = feasibilityModelByRule[rule]
    scored = reactionDF.copy()
    rxnStr = scored["reactionString"].astype(str).str.replace(" ", "", regex=False)
    scored[feasibilityScoreCol] = rxnStr.apply(model.predict_proba)
    scored[feasibilityLabelCol] = rxnStr.apply(model.predict_label)
    return scored


### Process one starter directory

`processStarter` reads a single `*_Starter` folder's DORAnet outputs and returns
the all possible and high feasibility compounds for that starter. 

In [30]:
def toMasterSchema(richDF, starterName, withFeasibility=False):
    """Map a rich per-starter frame to the master output columns."""
    baseCols = ["Starter_Name", "Starter_Canonical_SMILES", "Target_Canonical_SMILES", "DORAnet_gen"]
    if richDF.empty:
        return pd.DataFrame(columns=baseCols + ([feasibilityScoreCol] if withFeasibility else []))
    out = pd.DataFrame({
        "Starter_Name": starterName,
        "Starter_Canonical_SMILES": richDF["starter_Canonical_SMILES"],
        "Target_Canonical_SMILES": richDF["Canonical_SMILES"],
        "DORAnet_gen": richDF["generationSteps"],
    })
    if withFeasibility:
        out[feasibilityScoreCol] = richDF[feasibilityScoreCol]
    return out


def findMoleculesCsv(starterDir):
    """Locate the per-starter molecules CSV. The precursor starter_* folders use the
    '*_molecules.csv' convention; the flat Basidalin folder uses '*_uniqueMolecules.csv',
    so both spellings are accepted, with '*_molecules.csv' preferred when present."""
    matches = sorted(glob.glob(os.path.join(starterDir, "*_molecules.csv")))
    if not matches:
        matches = sorted(glob.glob(os.path.join(starterDir, "*_uniqueMolecules.csv")))
    return matches


def readStarterSmiles(molMatches):
    """Recover the starter SMILES from a molecules CSV's Is_Starter column."""
    starters = set()
    for molCsv in molMatches:
        tempDF = pd.read_csv(molCsv, usecols=["SMILES", "Is_Starter"])
        starters.update(tempDF[tempDF["Is_Starter"] == True]["SMILES"].dropna().tolist())
    return starters


def resolveFeasibilityTable(starterDir, reactionDF):
    """Return the DORA-XGB feasibility table for one starter.

    Looks first for a per-starter file inside the starter directory (the Inosine layout).
    If absent, falls back to a shared file one level up (the generatedMolecules layout,
    where one table covers every starter_* and carries a 'SourceDirectory' column),
    filtered to this starter. If neither exists (the Basidalin single-starter run, which
    ships no feasibility CSV), the network is scored live with DORA-XGB."""
    localPath = os.path.join(starterDir, feasibilityFileName)
    if os.path.isfile(localPath):
        return pd.read_csv(localPath)

    for sharedDir in (os.path.dirname(starterDir), BasidalinprecursorOutputDir):
        sharedPath = os.path.join(sharedDir, feasibilityFileName)
        if os.path.isfile(sharedPath):
            shared = pd.read_csv(sharedPath)
            if "SourceDirectory" in shared.columns:
                return shared[shared["SourceDirectory"] == os.path.basename(starterDir)].reset_index(drop=True)
            return shared

    print(f"[info] {os.path.basename(starterDir)}: no {feasibilityFileName}; scoring feasibility live ({feasibilityLabelCol.split('_')[-1]})")
    return computeFeasibilityLive(reactionDF)


def processStarter(starterDir, starterName, starterSmilesOverride=None):
    jsonMatches = sorted(glob.glob(os.path.join(starterDir, "*_network_pretreated.json")))
    molMatches = findMoleculesCsv(starterDir)
    if not jsonMatches:
        print(f"[skip] {starterName}: missing network json")
        return None, None

    # Starter SMILES: an explicit override (Basidalin, whose CSV Is_Starter is unreliable)
    # takes precedence; otherwise recover them from the molecules CSV Is_Starter column.
    if starterSmilesOverride:
        userStarters = {starterSmilesOverride}
    else:
        if not molMatches:
            print(f"[skip] {starterName}: missing molecules csv")
            return None, None
        userStarters = readStarterSmiles(molMatches)
    if not userStarters:
        print(f"[skip] {starterName}: no starter rows found")
        return None, None

    reactionDF = readReactionDf(jsonMatches[0])
    if reactionDF.empty or "products" not in reactionDF.columns:
        print(f"[skip] {starterName}: empty or unparseable reaction network")
        return None, None

    # Network depth for this starter, read from its doranet_config.yaml.
    generations = readGenerationsFromConfig(starterDir, defaultGenerations)

    # Save the parsed reactions for this starter (reaction pathways).
    reactionPath = os.path.join(resultsDir, f"{starterName}_reaction_pathways.csv")
    reactionDF.to_csv(reactionPath, index=False)

    # Per-starter sets.
    starterMols = [
        (canonicalizeSmiles(smi), Chem.MolFromSmiles(str(smi)))
        for smi in userStarters if Chem.MolFromSmiles(str(smi)) is not None
    ]
    canonStarterSet = {s for s, _ in starterMols}
    canonHelperSet = {canonicalizeSmiles(s) for s in helpersToExclude}
    starterFpByCanon = {
        s: morganFp(mol) for s, mol in starterMols if morganFp(mol) is not None
    }

    # All-possible: BFS over the full network.
    starterDerivedSet = buildStarterDerivedSet(
        reactionDF, starterMols, canonStarterSet, canonHelperSet,
        cofactorExclusionSet, physicochemicalCutoffs, starterFpByCanon,
    )
    allProducts = reactionDF["products"].apply(splitMoleculeString).explode().dropna().unique().tolist()
    allRaw = enumerateStarterDerivedProducts(
        allProducts, starterDerivedSet, canonHelperSet, canonStarterSet,
        cofactorExclusionSet, physicochemicalCutoffs,
    )
    generatedCompoundsDF = dedupLipinskiSort(allRaw)

    # Save the full per-starter generated compounds (rich columns).
    generatedPath = os.path.join(resultsDir, f"{starterName}_generated_compounds.csv")
    generatedCompoundsDF.to_csv(generatedPath, index=False)

    allOut = toMasterSchema(generatedCompoundsDF, starterName, withFeasibility=False)

    # High-feasibility: uses a per-starter CSV, a shared CSV, or live DORA-XGB scoring.
    reactionDF_DORAXGB = resolveFeasibilityTable(starterDir, reactionDF)
    if reactionDF_DORAXGB is None or reactionDF_DORAXGB.empty:
        print(f"[warn] {starterName}: no feasibility scores available; high-feasibility skipped")
        hfOut = pd.DataFrame(columns=list(allOut.columns) + [feasibilityScoreCol])
        starterContexts[starterName] = {
            "generations": generations,
            "canonStarterSet": canonStarterSet,
            "canonHelperSet": canonHelperSet,
            "reactionDF_highFeasible": None,
            "generatedCompoundsDF_highFeasibility": None,
        }
        print(f"[done] {starterName}: allPossible={len(allOut)}, highFeasibility=0")
        return allOut, hfOut

    reactionDF_highFeasible = ensureReactantProductCols(
        reactionDF_DORAXGB[
            (reactionDF_DORAXGB[feasibilityLabelCol] == feasibilityTargetValue)
            & (reactionDF_DORAXGB[feasibilityLabelCol].notna())
        ]
    )

    starterDerivedSetHF = buildStarterDerivedSet(
        reactionDF_highFeasible, starterMols, canonStarterSet, canonHelperSet,
        cofactorExclusionSet, physicochemicalCutoffs, starterFpByCanon,
    )
    hfProducts = reactionDF_highFeasible["products"].apply(splitMoleculeString).explode().dropna().unique().tolist()
    hfRaw = enumerateStarterDerivedProducts(
        hfProducts, starterDerivedSetHF, canonHelperSet, canonStarterSet,
        cofactorExclusionSet, physicochemicalCutoffs,
    )
    generatedCompoundsDF_highFeasibility = dedupLipinskiSort(hfRaw)

    # Preserve the full-network generation depth rather than the sparser subnetwork's.
    stepsByCompound = generatedCompoundsDF.drop_duplicates("Canonical_SMILES").set_index("Canonical_SMILES")["generationSteps"]
    if not generatedCompoundsDF_highFeasibility.empty:
        generatedCompoundsDF_highFeasibility["generationSteps"] = (
            generatedCompoundsDF_highFeasibility["Canonical_SMILES"].map(stepsByCompound)
        )

    # Per-compound feasibility score: max rule3 score among producing reactions.
    scoreByCompound = buildFeasibilityScoreByCompound(reactionDF_highFeasible, feasibilityScoreCol)
    if not generatedCompoundsDF_highFeasibility.empty:
        generatedCompoundsDF_highFeasibility[feasibilityScoreCol] = (
            generatedCompoundsDF_highFeasibility["Canonical_SMILES"].map(scoreByCompound)
        )

    # Save the per-starter high-feasibility compounds (rich columns + feasibility score).
    highFeasPath = os.path.join(resultsDir, f"{starterName}_generated_compounds_highFeasibility.csv")
    generatedCompoundsDF_highFeasibility.to_csv(highFeasPath, index=False)

    hfOut = toMasterSchema(generatedCompoundsDF_highFeasibility, starterName, withFeasibility=True)

    # Retain everything the pathway-plotting cells need for this starter.
    starterContexts[starterName] = {
        "generations": generations,
        "canonStarterSet": canonStarterSet,
        "canonHelperSet": canonHelperSet,
        "reactionDF_highFeasible": reactionDF_highFeasible,
        "generatedCompoundsDF_highFeasibility": generatedCompoundsDF_highFeasibility,
    }

    print(f"[done] {starterName}: allPossible={len(allOut)}, highFeasibility={len(hfOut)}")
    return allOut, hfOut


# Step 3B: Aggregate DORAnet-generated antibiotics across both Basidalin sources

This notebook post-processes the DORAnet runs under `Step3A_Antibiotic_Compounds/`.
For every starter it traces which molecules are derived from that starter, filters out
cofactors and non-drug-like products, and separates the high-feasibility subset using
the DORA-XGB (rule3) reaction scores. Per-starter files are written to `resultsDir`, and
the two sources are then combined into two master tables.

The two sources have different on-disk layouts and are processed in separate cells so the
workflows stay independent:

- `Step2_Basidalin` — a single flat starter directory for the basidalin scaffold itself
  (the `BasidalinOnly_gen3` run), read the same way as the Inosine starter. Its
  `Is_Starter` column is not reliable, so the starter SMILES is supplied explicitly.
- `Step2_BasidalinPrecursor/doranet_output/starter_*` — the many precursor starters, one
  subdirectory each, matching the generatedMolecules layout.

### Cell order

1. **Setup cell** — defines the memoized helpers and `processStarter`. Molecule
   canonicalization, cofactor test, fingerprint, Lipinski check, and molecular weight are
   each computed once per unique molecule and reused across reactions, across the
   all-possible and high-feasibility passes, and across every starter a worker handles.
   The cached values come from the same RDKit calls as before, so results are unchanged;
   only the repeated recomputation is removed. Run this before either processing cell.
2. **Cell 1 — Step2_Basidalin** — processes the single flat starter serially and collects
   its result into `basidalinAllFrames` / `basidalinHfFrames`.
3. **Cell 2 — Step2_BasidalinPrecursor** — processes every `starter_*` directory in
   parallel across up to four worker processes, collecting into
   `precursorAllFrames` / `precursorHfFrames`. A bad or empty network is recorded and
   skipped rather than aborting the batch, and a `tqdm` bar shows live kept / empty /
   failed counts.
4. **Cell 3 — Aggregate** — concatenates whichever source frames exist, deduplicates on
   `(Starter_Name, Target_Canonical_SMILES)`, and writes the two master tables.

Both processing cells write into a shared `starterContexts` used by the pathway-plotting
cells, and a guard lets either cell run first or on its own without clearing the other's
contexts.

### Per-starter inputs

For each starter directory `processStarter` reads the reaction network
(`*_network_pretreated.json`) and, to identify the starter, the molecules CSV
(`*_molecules.csv`, or `*_uniqueMolecules.csv` for the flat Basidalin run). The
all-possible list is built entirely from the network JSON; `reactionDF.csv`, if present,
is not used here. High-feasibility uses `reactionDF_wDORAXGBfeasibility.csv` when it exists
in the directory (or a shared copy one level up), and otherwise scores the network live.

### Master tables

Two tables are written to `resultsDir`:

- **All possible target compounds** — one row per (starter, derived compound):
  `Starter_Name`, `Starter_Canonical_SMILES`, `Target_Canonical_SMILES`, `DORAnet_gen`.
- **High-feasibility target compounds** — the same columns plus `feasibilityScore_rule3`.

The pathogen and paths come from the standard configuration block, so the notebook runs
for any target by changing `target`.

In [36]:
# Two DORAnet sources under Step3A_Antibiotic_Compounds.
precursorDoranetOutputDir = os.path.join(compoundsRootDir, "Step2_BasidalinPrecursor", "doranet_output")
basidalinStarterDir = os.path.join(compoundsRootDir, "Step2_Basidalin")
basidalinStarterName = "Basidalin"
basidalinStarterSmiles = r"C1=C(/C(=C\C=O)/OC1=O)N"

In [34]:
# Process Basidalin only compounds
# It is one starter, so no worker pool is needed.

try:
    starterContexts
except NameError:
    starterContexts = {}

basidalinAllFrames, basidalinHfFrames = [], []

if os.path.isdir(basidalinStarterDir):
    print(f"Processing flat starter: {basidalinStarterName} "
          f"-> {os.path.relpath(basidalinStarterDir, compoundsRootDir)}")
    allOut, hfOut = processStarter(basidalinStarterDir, basidalinStarterName, basidalinStarterSmiles)
    if allOut is not None and not allOut.empty:
        basidalinAllFrames.append(allOut)
    if hfOut is not None and not hfOut.empty:
        basidalinHfFrames.append(hfOut)
    nAll = len(basidalinAllFrames[0]) if basidalinAllFrames else 0
    nHf = len(basidalinHfFrames[0]) if basidalinHfFrames else 0
    print(f"Basidalin: allPossible={nAll:,}, highFeasibility={nHf:,}")
else:
    print(f"Step2_Basidalin directory not found: {basidalinStarterDir}")

Processing flat starter: Basidalin -> Step2_Basidalin
[warn] Step2_Basidalin: no doranet_config.yaml; using generations=3
[info] Step2_Basidalin: no reactionDF_wDORAXGBfeasibility.csv; scoring feasibility live (rule3)
[done] Basidalin: allPossible=4078, highFeasibility=515
Basidalin: allPossible=4,078, highFeasibility=515


In [37]:
# process all Step2_BasidalinPrecursor starter_* directories in parallel

import io
import multiprocessing
from contextlib import redirect_stdout
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

try:
    starterContexts
except NameError:
    starterContexts = {}

# Up to 4 workers, never more than the machine has.
numWorkers = min(4, os.cpu_count() or 1)

# Keep each starter's plotting context. Set False for a very large batch to save memory
# and inter-process transfer; the pathway-plotting cell then only works for starters you
# reprocess afterward.
keepPlotContexts = True


def runStarterJob(job):
    """Worker entry point: process one precursor starter and return its outputs plus context.
    Runs in a forked child, so it reuses the parent's globals and the per-worker caches
    warm up across the many starters each worker handles."""
    starterDir, starterName, starterSmilesOverride = job
    with redirect_stdout(io.StringIO()):
        allOut, hfOut = processStarter(starterDir, starterName, starterSmilesOverride)
    context = starterContexts.get(starterName) if keepPlotContexts else None
    return starterName, allOut, hfOut, context


precursorJobs = [
    (subDir, f"BasidalinPrecursor_{os.path.basename(subDir)}", None)
    for subDir in sorted(glob.glob(os.path.join(precursorDoranetOutputDir, "starter_*")))
    if os.path.isdir(subDir)
]
print(f"Found {len(precursorJobs)} precursor starter directories")

precursorAllFrames, precursorHfFrames = [], []
nKept = nEmpty = nFailed = 0
failures = []

forkContext = multiprocessing.get_context("fork")
print(f"Processing {len(precursorJobs)} precursor starters on {numWorkers} workers")
with ProcessPoolExecutor(max_workers=numWorkers, mp_context=forkContext) as executor:
    futureToName = {executor.submit(runStarterJob, job): job[1] for job in precursorJobs}

    progressBar = tqdm(as_completed(futureToName), total=len(futureToName),
                       desc="Precursor starters", unit="starter")
    for future in progressBar:
        starterName = futureToName[future]
        try:
            name, allOut, hfOut, context = future.result()
        except Exception as exc:
            nFailed += 1
            if len(failures) < 10:
                failures.append((starterName, str(exc)))
            progressBar.set_postfix(kept=nKept, empty=nEmpty, failed=nFailed, refresh=False)
            continue

        if context is not None:
            starterContexts[name] = context
        if allOut is not None and not allOut.empty:
            precursorAllFrames.append(allOut)
            nKept += 1
        else:
            nEmpty += 1
        if hfOut is not None and not hfOut.empty:
            precursorHfFrames.append(hfOut)

        progressBar.set_postfix(kept=nKept, empty=nEmpty, failed=nFailed, refresh=False)

print(f"\nPrecursor starters kept: {nKept:,} | empty or skipped: {nEmpty:,} | failed: {nFailed:,}")
if failures:
    print("First failures:")
    for name, msg in failures:
        print(f"  {name}: {msg}")

Found 2054 precursor starter directories
Processing 2054 precursor starters on 4 workers


Precursor starters: 100%|█████████████████████████████████████████| 2054/2054 [16:32<00:00,  2.07starter/s, empty=565, failed=0, kept=1489]


Precursor starters kept: 1,489 | empty or skipped: 565 | failed: 0


In [38]:
# Aggregate Basidalin + BasidalinPrecursor into the master tables and save

allFrames = (globals().get("basidalinAllFrames") or []) + (globals().get("precursorAllFrames") or [])
hfFrames  = (globals().get("basidalinHfFrames") or []) + (globals().get("precursorHfFrames") or [])

outColumns = ["Starter_Name", "Starter_Canonical_SMILES", "Target_Canonical_SMILES", "DORAnet_gen"]

masterAllPossible = (
    pd.concat(allFrames, ignore_index=True)
    .drop_duplicates(subset=["Starter_Name", "Target_Canonical_SMILES"])
    .reset_index(drop=True)
    if allFrames else pd.DataFrame(columns=outColumns)
)
masterHighFeasibility = (
    pd.concat(hfFrames, ignore_index=True)
    .drop_duplicates(subset=["Starter_Name", "Target_Canonical_SMILES"])
    .reset_index(drop=True)
    if hfFrames else pd.DataFrame(columns=outColumns + [feasibilityScoreCol])
)

allPath = os.path.join(resultsDir, f"{target}_antibiotics_allPossible.csv")
hfPath = os.path.join(resultsDir, f"{target}_antibiotics_highFeasibility.csv")
masterAllPossible.to_csv(allPath, index=False)
masterHighFeasibility.to_csv(hfPath, index=False)

print(f"All possible target compounds    : {len(masterAllPossible):,} rows -> {allPath}")
print(f"High-feasibility target compounds: {len(masterHighFeasibility):,} rows -> {hfPath}")
print("\nTop starters by compound count (all possible):")
if not masterAllPossible.empty:
    print(masterAllPossible["Starter_Name"].value_counts().head(10).to_string())

All possible target compounds    : 995,697 rows -> /users/sghosh6/DTRA_project/MACAW/SynThera/BacillusAnthracis/Results_BacillusAnthracis/BacillusAnthracis_antibiotics_allPossible.csv
High-feasibility target compounds: 101,838 rows -> /users/sghosh6/DTRA_project/MACAW/SynThera/BacillusAnthracis/Results_BacillusAnthracis/BacillusAnthracis_antibiotics_highFeasibility.csv

Top starters by compound count (all possible):
Starter_Name
BasidalinPrecursor_starter_01899    4286
BasidalinPrecursor_starter_01937    4144
BasidalinPrecursor_starter_01973    4143
BasidalinPrecursor_starter_01923    4142
BasidalinPrecursor_starter_01964    4141
Basidalin                           4078
BasidalinPrecursor_starter_02036    4025
BasidalinPrecursor_starter_02039    4005
BasidalinPrecursor_starter_02008    3808
BasidalinPrecursor_starter_02025    3808


## Plot pathways

The pathway renderers below reproduce the original notebook. They operate on one
starter at a time, so choose a starter here; the cell wires up the exact
variables the plotting cells expect, using that starter's own network depth
(`generations`) read from its `doranet_config.yaml`.

In [ ]:
# Choose which starter's pathways to plot. Change this to any discovered starter.
starterToPlot = "Basidalin"

if starterToPlot not in starterContexts:
    raise KeyError(f"'{starterToPlot}' not found. Available: {sorted(starterContexts)}")

_ctx = starterContexts[starterToPlot]
if _ctx["reactionDF_highFeasible"] is None:
    raise ValueError(
        f"'{starterToPlot}' has no high-feasibility data (missing {feasibilityFileName}); cannot plot."
    )

# Variables the original plotting cells expect.
reactionDF_highFeasible = _ctx["reactionDF_highFeasible"]
reactionDF_DORAXGB_highFeasibility = _ctx["reactionDF_highFeasible"]
generatedCompoundsDF_highFeasibility = _ctx["generatedCompoundsDF_highFeasibility"]
canonStarterSet = _ctx["canonStarterSet"]
canonHelperSet = _ctx["canonHelperSet"]
doranet_generations = _ctx["generations"]
bestRuleScoreCol = feasibilityScoreCol
fileNamePrefix = starterToPlot

print(f"Plotting pathways for {starterToPlot} (network depth {doranet_generations})")
print(f"  high-feasibility reactions: {len(reactionDF_highFeasible):,}")
print(f"  high-feasibility compounds: {len(generatedCompoundsDF_highFeasibility):,}")

### Feasible pathways (vertical, per reaction step)

In [ ]:
# CONFIG
numToDisplayPerStep = 4
saveFigures         = False
figureOutDir        = os.path.join(resultsDir, "PathwayFigures")
subImgSize          = (600, 300)
articleFiguresDir     = os.path.join(resultsDir, "PathwayFigures")
rulesetPath           = rulesetPathForPlots
cofactorTablePath     = cofactorTablePathForPlots
gapSize=14

cofactorDisplayOverride = {
    "CO-A": "CoA", "ACETYL-COA": "Acetyl-CoA", "OXYGEN-MOLECULE": "O2",
    "WATER": "H2O", "HYDROGEN-PEROXIDE": "H2O2", "CARBON-DIOXIDE": "CO2",
    "AMMONIA": "NH3", "S-ADENOSYLMETHIONINE": "SAM", "ADENOSYL-HOMO-CYS": "SAH",
    "2-KETOGLUTARATE": "2-Ketoglutarate", "GLT": "Glutamate", "PROTON": "H+",
}

def loadFont(size):
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

separatorFont = loadFont(52)
labelFont     = loadFont(18)
headerFont    = loadFont(22)

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

def cofactorKey(smi):
    mol = Chem.MolFromSmiles(str(smi), sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except Exception:
        return None

def loadCofactorNames(path):
    nameByKey = {}
    if not os.path.exists(path):
        print(f"Cofactor table not found at {path}; cofactor names will not be applied.")
        return nameByKey
    table = pd.read_csv(path, sep="\t")
    if "SMILES" not in table.columns or "Name" not in table.columns:
        print(f"Cofactor table {path} lacks Name/SMILES columns; names not applied.")
        return nameByKey
    for _, row in table.iterrows():
        key = cofactorKey(row["SMILES"])
        if key is None:
            continue
        rawName = str(row["Name"]).strip()
        nameByKey.setdefault(key, cofactorDisplayOverride.get(rawName, rawName))
    return nameByKey

cofactorNameByKey = loadCofactorNames(cofactorTablePath)

def cofactorNameFor(smi):
    return cofactorNameByKey.get(cofactorKey(smi))

def rdkitImageToPil(imageObj):
    if isinstance(imageObj, Image.Image):
        return imageObj.convert("RGB")
    if isinstance(imageObj, (bytes, bytearray)):
        return Image.open(BytesIO(imageObj)).convert("RGB")
    return Image.open(BytesIO(imageObj.data)).convert("RGB")

def centeredText(draw, text, imgWidth, y, font, color="black"):
    try:
        bbox  = draw.textbbox((0, 0), text, font=font)
        textW = bbox[2] - bbox[0]
    except Exception:
        textW = len(text) * 8
    draw.text(((imgWidth - textW) // 2, y), text, fill=color, font=font)

def makeTextBanner(text, bannerWidth, bannerHeight=40, bgColor="white", font=None):
    img = Image.new("RGB", (bannerWidth, bannerHeight), bgColor)
    centeredText(ImageDraw.Draw(img), text, bannerWidth, (bannerHeight - 24) // 2, font or headerFont)
    return img

def stackPanelsVertically(panelList, gapSize=14, bgColor="white"):
    valid = [p for p in panelList if p is not None]
    if not valid:
        return None
    w = max(p.width for p in valid)
    h = sum(p.height for p in valid) + gapSize * (len(valid) - 1)
    canvas = Image.new("RGB", (w, h), bgColor)
    y = 0
    for p in valid:
        canvas.paste(p, ((w - p.width) // 2, y))
        y += p.height + gapSize
    return canvas

def saveImageHighRes(image, basePathNoExt, dpi):
    pngPath = basePathNoExt + ".png"
    pdfPath = basePathNoExt + ".pdf"
    image.save(pngPath, dpi=(dpi, dpi))
    image.save(pdfPath, "PDF", resolution=float(dpi))
    return pngPath, pdfPath

def reorderStepsForward(steps, canonStarterSet):
    if len(steps) <= 1:
        return steps
    parsed = []
    for smi in steps:
        if ">>" not in str(smi):
            continue
        rStr, pStr = str(smi).split(">>", 1)
        parsed.append({
            "smi"      : smi,
            "reactants": {canonicalizeSmiles(s.strip()) for s in rStr.split(".") if s.strip()},
            "products" : {canonicalizeSmiles(s.strip()) for s in pStr.split(".") if s.strip()},
        })
    if not parsed:
        return steps
    firstIdx = next((i for i, s in enumerate(parsed) if s["reactants"] & canonStarterSet), None)
    if firstIdx is None:
        return steps
    ordered   = [parsed[firstIdx]]
    remaining = [s for i, s in enumerate(parsed) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["products"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactants"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return [s["smi"] for s in ordered]

def getMolLabel(smi, canonStarterSet, canonTargetSmi, side="reactant"):
    canon = canonicalizeSmiles(smi)
    if canon in canonStarterSet:
        return "Starter compound"
    if canon == canonTargetSmi:
        return "Target compound"
    cofactorName = cofactorNameFor(smi)
    if cofactorName:
        return cofactorName
    return "Reactant" if side == "reactant" else "Product"

def makeLabeledMolImage(smi, label, subImageSize=(280, 210)):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    molImg      = rdkitImageToPil(Draw.MolToImage(mol, size=subImageSize))
    labelBanner = Image.new("RGB", (molImg.width, 34), "#f0f0f0")
    centeredText(ImageDraw.Draw(labelBanner), label, molImg.width, 7, labelFont, "#333333")
    combined = Image.new("RGB", (molImg.width, molImg.height + labelBanner.height), "white")
    combined.paste(molImg,      (0, 0))
    combined.paste(labelBanner, (0, molImg.height))
    return combined

def makeSeparatorImage(text, height, width=90):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        bbox         = draw.textbbox((0, 0), text, font=separatorFont)
        textW, textH = bbox[2] - bbox[0], bbox[3] - bbox[1]
    except Exception:
        textW, textH = 30, 50
    draw.text(((width - textW) // 2, (height - textH) // 2), text, fill="#222222", font=separatorFont)
    return img

def makeCustomReactionPanel(rxnSmi, canonStarterSet, canonTargetSmi,
                             panelTitle, subImageSize=(280, 210)):
    if ">>" not in str(rxnSmi):
        return None
    reactantStr, productStr = str(rxnSmi).split(">>", 1)
    reactantSmiles = (
        [s for s in [x.strip() for x in reactantStr.split(".") if x.strip()] if canonicalizeSmiles(s) in canonStarterSet]
        + [s for s in [x.strip() for x in reactantStr.split(".") if x.strip()] if canonicalizeSmiles(s) not in canonStarterSet]
    )
    productSmiles = (
        [s for s in [x.strip() for x in productStr.split(".") if x.strip()] if canonicalizeSmiles(s) != canonTargetSmi]
        + [s for s in [x.strip() for x in productStr.split(".") if x.strip()] if canonicalizeSmiles(s) == canonTargetSmi]
    )
    reactantImgs = [img for img in [
        makeLabeledMolImage(s, getMolLabel(s, canonStarterSet, canonTargetSmi, side="reactant"), subImageSize)
        for s in reactantSmiles] if img is not None]
    productImgs  = [img for img in [
        makeLabeledMolImage(s, getMolLabel(s, canonStarterSet, canonTargetSmi, side="product"), subImageSize)
        for s in productSmiles]  if img is not None]
    if not reactantImgs or not productImgs:
        return None
    maxH  = max(img.height for img in reactantImgs + productImgs)
    gap   = 16
    parts = []
    for i, img in enumerate(reactantImgs):
        parts.append(img)
        if i < len(reactantImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=80))
    parts.append(makeSeparatorImage("→", maxH, width=100))
    for i, img in enumerate(productImgs):
        parts.append(img)
        if i < len(productImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=80))
    totalW = sum(p.width for p in parts) + gap * (len(parts) - 1)
    canvas = Image.new("RGB", (totalW, maxH), "white")
    x = 0
    for part in parts:
        canvas.paste(part, (x, (maxH - part.height) // 2))
        x += part.width + gap
    titleBanner = makeTextBanner(panelTitle, canvas.width, bannerHeight=38)
    final = Image.new("RGB", (canvas.width, canvas.height + titleBanner.height + 10), "white")
    final.paste(titleBanner, (0, 0))
    final.paste(canvas,      (0, titleBanner.height + 10))
    return final

# Feasible-reaction network (all steps here are DORA-XGB high-feasibility by construction)
feasibleRxnDF = reactionDF_highFeasible.copy()
if "reactionString" not in feasibleRxnDF.columns:
    feasibleRxnDF["reactionString"] = (
        feasibleRxnDF["reactants"].astype(str) + " >> " + feasibleRxnDF["products"].astype(str)
    )

# canonical reactant -> feasible reactions consuming it
reactantToFeasibleRxns = defaultdict(list)
for idx, row in feasibleRxnDF.iterrows():
    for smi in splitMoleculeString(row["reactants"]):
        reactantToFeasibleRxns[canonicalizeSmiles(smi)].append(idx)

def carrierProductsOf(row):
    """Non-cofactor, non-helper, non-starter products that can be transformed further."""
    out = []
    for productSmi in splitMoleculeString(row["products"]):
        productCanon = canonicalizeSmiles(productSmi)
        if productCanon in canonHelperSet:
            continue
        if productCanon in canonStarterSet:
            continue
        productMol = Chem.MolFromSmiles(str(productSmi))
        if isCofactorOrEndogenous(productMol, productCanon, cofactorExclusionSet, physicochemicalCutoffs):
            continue
        out.append(productCanon)
    return out

# Depth-capped BFS over feasible edges: shortest fully-feasible route (<= doranet_generations) to each compound.
# parent[compound] = (rxnIdx, precursorCanon); absence means "no all-feasible route within doranet_generations".
parentReaction = {}
seenCompounds  = set(canonStarterSet)
searchQueue    = deque((c, 0) for c in canonStarterSet)

while searchQueue:
    currentCanon, currentDepth = searchQueue.popleft()
    if currentDepth >= doranet_generations:
        continue
    for rxnIdx in reactantToFeasibleRxns.get(currentCanon, []):
        row = feasibleRxnDF.loc[rxnIdx]
        for productCanon in carrierProductsOf(row):
            if productCanon in seenCompounds:
                continue
            seenCompounds.add(productCanon)
            parentReaction[productCanon] = (rxnIdx, currentCanon)
            searchQueue.append((productCanon, currentDepth + 1))

def feasibleRouteTo(targetCanon):
    """Reaction strings of the fully-feasible route to a target, forward order; [] if none within doranet_generations."""
    steps = []
    cur   = targetCanon
    while cur in parentReaction:
        rxnIdx, precursorCanon = parentReaction[cur]
        steps.append(feasibleRxnDF.loc[rxnIdx, "reactionString"])
        cur = precursorCanon
    steps.reverse()
    return steps

def reorderStepsForward(steps, canonStarterSet):
    if len(steps) <= 1:
        return steps
    parsed = []
    for smi in steps:
        if ">>" not in str(smi):
            continue
        rStr, pStr = str(smi).split(">>", 1)
        parsed.append({
            "smi"      : smi,
            "reactants": {canonicalizeSmiles(s.strip()) for s in rStr.split(".") if s.strip()},
            "products" : {canonicalizeSmiles(s.strip()) for s in pStr.split(".") if s.strip()},
        })
    if not parsed:
        return steps
    firstIdx = next((i for i, s in enumerate(parsed) if s["reactants"] & canonStarterSet), None)
    if firstIdx is None:
        return steps
    ordered   = [parsed[firstIdx]]
    remaining = [s for i, s in enumerate(parsed) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["products"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactants"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return [s["smi"] for s in ordered]

def buildRouteImageFromSteps(stepReactions, starterCanon, targetCanon,
                             headerText, subImageSize, bannerWidth):
    canonStarterSetLocal = {starterCanon}
    numSteps             = len(stepReactions)
    forwardSteps         = reorderStepsForward(stepReactions, canonStarterSetLocal)
    panelBlocks = [makeTextBanner(headerText, bannerWidth=bannerWidth, bannerHeight=44, bgColor="#eef2f7")]
    for stepIdx, rxnSmi in enumerate(forwardSteps, start=1):
        panel = makeCustomReactionPanel(
            rxnSmi, canonStarterSetLocal, targetCanon,
            panelTitle=f"Reaction step {stepIdx} of {numSteps}",
            subImageSize=subImageSize,
        )
        if panel:
            panelBlocks.append(panel)
    return stackPanelsVertically(panelBlocks, gapSize=14, bgColor="white")

if saveFigures:
    os.makedirs(figureOutDir, exist_ok=True)

candidateDF = generatedCompoundsDF_highFeasibility.reset_index(drop=True)

# Keep only candidates that have a fully-feasible route within doranet_generations
feasibleRoutes = []
for _, cand in candidateDF.iterrows():
    targetCanon = canonicalizeSmiles(cand["Canonical_SMILES"])
    steps       = feasibleRouteTo(targetCanon)
    if steps and len(steps) <= doranet_generations:
        firstReactants = str(steps[0]).split(">>", 1)[0]
        starterCanon   = next(
            (canonicalizeSmiles(s) for s in firstReactants.split(".") if canonicalizeSmiles(s) in canonStarterSet),
            cand.get("starter_Canonical_SMILES", ""),
        )
        feasibleRoutes.append((len(steps), targetCanon, starterCanon, steps))

feasibleRoutes.sort(key=lambda t: t[0])

print(f"High-feasibility candidates                 : {len(candidateDF):,}")
print(f"With a fully-feasible route (<= {doranet_generations} steps) : {len(feasibleRoutes):,}")
stepHistogram = defaultdict(int)
for nSteps, *_ in feasibleRoutes:
    stepHistogram[nSteps] += 1
for nSteps in sorted(stepHistogram):
    print(f"  {nSteps}-step feasible pathways : {stepHistogram[nSteps]:,}")

drawnByStep = defaultdict(int)
for numSteps, targetCanon, starterCanon, steps in feasibleRoutes:
    if drawnByStep[numSteps] >= numToDisplayPerStep:
        continue
    drawnByStep[numSteps] += 1
    headerText = f"Starter: {str(starterCanon)[:40]}  |  Target: {targetCanon[:50]}  |  Steps: {numSteps}"
    routeImage = buildRouteImageFromSteps(steps, starterCanon, targetCanon, headerText, subImgSize, bannerWidth=1800)
    if routeImage is None:
        continue
    print(f"  [{numSteps}-step #{drawnByStep[numSteps]}] {targetCanon}")
    display(routeImage)
    if saveFigures:
        basePath         = os.path.join(figureOutDir, f"highFeasPathway_{numSteps}step_{drawnByStep[numSteps]}")
        pngPath, pdfPath = saveImageHighRes(routeImage, basePath, outputDpi)
        print(f"      saved: {pngPath}")

print("\nDone.")

### Feasible pathways (horizontal cascade with feasibility scores)

In [ ]:
# CONFIG
numToDisplay           = 5
selectedRoutes         = {}   # empty: every step-count group shows its first numToDisplay routes
layoutDirection        = "horizontal"
moleculeImageSizePx    = (900, 680)
arrowSpanPx            = 520
outputDpi              = 1200
showStarterTargetTags  = True
showCofactorLabels     = True
showFeasibilityScore   = True
feasibilityScoreColumn = bestRuleScoreCol   # column in reactionDF_DORAXGB_highFeasibility
feasibilityScorePrefix = ""                          # set to "feasibility = " to prefix the number
saveFigures            = True
maxCofactorHeavyAtoms  = 12   # unknown species larger than this are treated as co-products, not cofactors

# Optional display-name overrides, keyed by the Name in all_cofactors.tsv.
# Leave empty to show the table names verbatim (CO-A, OXYGEN-MOLECULE, ...).
cofactorDisplayOverride = {
    "CO-A": "CoA",
    "ACETYL-COA": "Acetyl-CoA",
    "OXYGEN-MOLECULE": "O2",
    "WATER": "H2O",
    "HYDROGEN-PEROXIDE": "H2O2",
    "CARBON-DIOXIDE": "CO2",
    "AMMONIA": "NH3",
    "S-ADENOSYLMETHIONINE": "SAM",
    "ADENOSYL-HOMO-CYS": "SAH",
    "2-KETOGLUTARATE": "2-Ketoglutarate",
    "GLT": "Glutamate",
    "PROTON": "H+",
}

def loadFont(size):
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

captionFont     = loadFont(30)
cofactorFont    = loadFont(26)
feasibilityFont = loadFont(28)

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

def cofactorKey(smi):
    """
    Canonical, stereo- and charge-insensitive key. Tolerates '*' wildcard atoms
    in the cofactor table (FAD, F420, quinones, formyl-THF).
    """
    mol = Chem.MolFromSmiles(str(smi), sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except Exception:
        return None

def loadCofactorNames(path):
    """Map canonical SMILES key -> display name, sourced from all_cofactors.tsv."""
    nameByKey = {}
    if not os.path.exists(path):
        print(f"Cofactor table not found at {path}; cofactors will fall back to formulas.")
        return nameByKey
    table = pd.read_csv(path, sep="\t")
    if "SMILES" not in table.columns or "Name" not in table.columns:
        print(f"Cofactor table {path} lacks Name/SMILES columns; names not applied.")
        return nameByKey
    for _, row in table.iterrows():
        key = cofactorKey(row["SMILES"])
        if key is None:
            continue
        rawName = str(row["Name"]).strip()
        nameByKey.setdefault(key, cofactorDisplayOverride.get(rawName, rawName))
    return nameByKey

cofactorNameByKey = loadCofactorNames(cofactorTablePath)

def isKnownCofactor(smi):
    return cofactorKey(smi) in cofactorNameByKey

def labelSpecies(smi):
    key = cofactorKey(smi)
    if key in cofactorNameByKey:
        return cofactorNameByKey[key]
    mol = Chem.MolFromSmiles(str(smi))
    return rdMolDescriptors.CalcMolFormula(mol) if mol else str(smi)

# Rule-level role tokens, used only when a reaction carries no explicit cofactor molecule.
cofactorTokenNameMap = {
    "WATER": "H2O", "HYDROGEN_PEROXIDE": "H2O2", "OXYGEN": "O2", "O2": "O2",
    "CARBON_DIOXIDE": "CO2", "AMMONIA": "NH3", "AMMONIUM": "NH4+",
    "NAD": "NAD+", "NAD_PLUS": "NAD+", "NADH": "NADH",
    "NADP": "NADP+", "NADP_PLUS": "NADP+", "NADPH": "NADPH",
    "L_GLUTAMATE": "Glutamate", "GLUTAMATE": "Glutamate",
    "2_KETOGLUTARATE": "2-Ketoglutarate", "2_OXOGLUTARATE": "2-Oxoglutarate",
    "METHYL_DONOR_COF": "SAM", "METHYL_ACCEPTOR_COF": "SAH",
    "ATP": "ATP", "ADP": "ADP", "AMP": "AMP",
    "PHOSPHATE": "Pi", "DIPHOSPHATE": "PPi", "COA": "CoA", "ACETYL_COA": "Acetyl-CoA",
}
primaryRoleTokens = {"ANY", "ANY_COF", ""}

def heavyAtomCount(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return mol.GetNumHeavyAtoms() if mol else 0

def splitStarters(starterStr):
    return [] if pd.isna(starterStr) else [x.strip() for x in str(starterStr).split(";") if x.strip()]

def dedupeByCanon(smilesList):
    seen, kept = set(), []
    for s in smilesList:
        if not s:
            continue
        c = canonicalizeSmiles(s)
        if c not in seen:
            seen.add(c)
            kept.append(s)
    return kept

def dedupePreserveOrder(items):
    seen, kept = set(), []
    for x in items:
        if x and x not in seen:
            seen.add(x)
            kept.append(x)
    return kept

def cleanRoleToken(token):
    key = token.strip().upper().replace("-", "_").replace(" ", "_")
    if key in cofactorTokenNameMap:
        return cofactorTokenNameMap[key]
    return token.strip().replace("_", " ").title()

def extractRuleCofactors(cellValue):
    tokens = [t.strip() for t in re.split(r"[;,]", str(cellValue)) if t.strip()]
    labels = [cleanRoleToken(t) for t in tokens if t.strip().upper() not in primaryRoleTokens]
    return dedupePreserveOrder(labels)

def buildRuleCofactorMap(path):
    ruleMap = {}
    if not os.path.exists(path):
        print(f"  ruleset not found at {path}, using reaction-level cofactors only.")
        return ruleMap
    ruleset = pd.read_csv(path, sep="\t")
    nameCol = "Name" if "Name" in ruleset.columns else ruleset.columns[0]
    hasReact, hasProd = "Reactants" in ruleset.columns, "Products" in ruleset.columns
    for _, row in ruleset.iterrows():
        name = str(row.get(nameCol, "")).strip()
        if not name:
            continue
        entry = {
            "consumed": extractRuleCofactors(row["Reactants"]) if hasReact else [],
            "produced": extractRuleCofactors(row["Products"]) if hasProd else [],
        }
        ruleMap.setdefault(name, entry)
        ruleMap.setdefault(name.split("_")[0], entry)
    return ruleMap

def ruleCofactorsForStep(ruleName, ruleCofactorMap):
    name = str(ruleName).strip()
    entry = ruleCofactorMap.get(name) or ruleCofactorMap.get(name.split("_")[0])
    return (entry["consumed"], entry["produced"]) if entry else ([], [])

def parseReactionSides(reactionSmiles):
    if ">>" not in str(reactionSmiles):
        return [], []
    rStr, pStr = str(reactionSmiles).split(">>", 1)
    return (
        [s.strip() for s in rStr.split(".") if s.strip()],
        [s.strip() for s in pStr.split(".") if s.strip()],
    )

def reactionKey(reactionSmiles):
    """Order- and spacing-independent reaction identity: canonical reactants and products."""
    reactants, products = parseReactionSides(reactionSmiles)
    reactantKey = ".".join(sorted(canonicalizeSmiles(s) for s in reactants))
    productKey  = ".".join(sorted(canonicalizeSmiles(s) for s in products))
    return reactantKey + ">>" + productKey

def buildFeasibilityScoreMap(scoreDF):
    """Map canonical reactionKey -> feasibilityScore_rule3 from reactionDF_DORAXGB_highFeasibility."""
    if feasibilityScoreColumn not in scoreDF.columns:
        raise KeyError(f"Column '{feasibilityScoreColumn}' not in reactionDF_DORAXGB_highFeasibility; available: {list(scoreDF.columns)}")
    if "reactionString" in scoreDF.columns:
        reactionSeries = scoreDF["reactionString"].astype(str)
    elif "reactants" in scoreDF.columns and "products" in scoreDF.columns:
        reactionSeries = scoreDF["reactants"].astype(str) + " >> " + scoreDF["products"].astype(str)
    else:
        raise KeyError("reactionDF_DORAXGB_highFeasibility needs a 'reactionString' column or both 'reactants' and 'products' columns to key reactions.")
    scoreByReaction = {}
    for reactionStr, score in zip(reactionSeries, scoreDF[feasibilityScoreColumn]):
        try:
            scoreValue = float(score)
        except (TypeError, ValueError):
            continue
        scoreByReaction.setdefault(reactionKey(reactionStr), scoreValue)
    return scoreByReaction

def reorderStepDictsForward(stepDicts, canonStarterSet):
    if len(stepDicts) <= 1:
        return stepDicts
    enriched = []
    for sd in stepDicts:
        reactants, products = parseReactionSides(sd["reactionSmiles"])
        if not reactants and not products:
            continue
        enriched.append({
            **sd,
            "reactantSet": {canonicalizeSmiles(s) for s in reactants},
            "productSet" : {canonicalizeSmiles(s) for s in products},
        })
    if not enriched:
        return stepDicts
    firstIdx = next((i for i, s in enumerate(enriched) if s["reactantSet"] & canonStarterSet), None)
    if firstIdx is None:
        return stepDicts
    ordered   = [enriched[firstIdx]]
    remaining = [s for i, s in enumerate(enriched) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["productSet"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactantSet"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return ordered

def pickCarrier(products, prevCarrierCanon, canonTargetSmi, nextReactants):
    # never let a known cofactor become a backbone node
    pool = [p for p in products if canonicalizeSmiles(p) != prevCarrierCanon and not isKnownCofactor(p)]
    if not pool:
        pool = [p for p in products if canonicalizeSmiles(p) != prevCarrierCanon] or products
    carrier = next((p for p in pool if canonicalizeSmiles(p) == canonTargetSmi), None)
    if carrier is None and nextReactants is not None:
        nextCanon = {canonicalizeSmiles(s) for s in nextReactants}
        feeding = [p for p in pool if canonicalizeSmiles(p) in nextCanon]
        if feeding:
            carrier = max(feeding, key=heavyAtomCount)
    if carrier is None:
        carrier = max(pool, key=heavyAtomCount)
    return carrier

def resolveCofactorLabels(step, reactants, products, prevCarrierCanon, carrierCanon, ruleCofactorMap):
    def keepAsCofactor(smi):
        return isKnownCofactor(smi) or heavyAtomCount(smi) <= maxCofactorHeavyAtoms

    consumedSpecies = [s for s in dedupeByCanon([r for r in reactants if canonicalizeSmiles(r) != prevCarrierCanon]) if keepAsCofactor(s)]
    producedSpecies = [s for s in dedupeByCanon([p for p in products  if canonicalizeSmiles(p) != carrierCanon])   if keepAsCofactor(s)]
    consumedLabels = [labelSpecies(s) for s in consumedSpecies]
    producedLabels = [labelSpecies(s) for s in producedSpecies]

    ruleConsumed, ruleProduced = ruleCofactorsForStep(step.get("ruleName", ""), ruleCofactorMap)
    if not consumedLabels and not producedLabels and (ruleConsumed or ruleProduced):
        consumedLabels, producedLabels = ruleConsumed, ruleProduced

    return dedupePreserveOrder(consumedLabels), dedupePreserveOrder(producedLabels)

def buildBackbone(stepDicts, canonStarterSet, canonTargetSmi, ruleCofactorMap):
    steps = []
    for sd in stepDicts:
        reactants, products = parseReactionSides(sd["reactionSmiles"])
        if not reactants or not products:
            continue
        steps.append({**sd, "reactants": reactants, "products": products})
    if not steps:
        return [], []

    firstReactants = steps[0]["reactants"]
    startCarrier = next((s for s in firstReactants if canonicalizeSmiles(s) in canonStarterSet), None)
    if startCarrier is None:
        nonCofactorFirst = [s for s in firstReactants if not isKnownCofactor(s)] or firstReactants
        startCarrier = max(nonCofactorFirst, key=heavyAtomCount)
    nodes            = [startCarrier]
    edges            = []
    prevCarrierCanon = canonicalizeSmiles(startCarrier)

    for stepIdx, step in enumerate(steps):
        reactants, products = step["reactants"], step["products"]
        nextReactants = steps[stepIdx + 1]["reactants"] if stepIdx + 1 < len(steps) else None
        carrier      = pickCarrier(products, prevCarrierCanon, canonTargetSmi, nextReactants)
        carrierCanon = canonicalizeSmiles(carrier)
        consumedLabels, producedLabels = resolveCofactorLabels(
            step, reactants, products, prevCarrierCanon, carrierCanon, ruleCofactorMap
        )
        nodes.append(carrier)
        edges.append({
            "consumed"         : consumedLabels,
            "produced"         : producedLabels,
            "feasibilityScore" : step.get("feasibilityScore"),
        })
        prevCarrierCanon = carrierCanon
    return nodes, edges

def drawMoleculeCairo(smi, sizePx):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    AllChem.Compute2DCoords(mol)
    drawer = rdMolDraw2D.MolDraw2DCairo(sizePx[0], sizePx[1])
    opts = drawer.drawOptions()
    opts.bondLineWidth   = 3
    opts.padding         = 0.10
    opts.clearBackground = True
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    return Image.open(BytesIO(drawer.GetDrawingText())).convert("RGB")

def textWidth(draw, text, font):
    try:
        bbox = draw.textbbox((0, 0), text, font=font)
        return bbox[2] - bbox[0]
    except Exception:
        return len(text) * 12

def centeredTextX(draw, text, boxWidth, y, font, color="#333333"):
    draw.text(((boxWidth - textWidth(draw, text, font)) // 2, y), text, fill=color, font=font)

def makeCaptionBanner(text, width, height=46):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    centeredTextX(draw, text, width, (height - 34) // 2, captionFont)
    return img

def makeMoleculePanel(smi, caption, imageSizePx):
    molImg = drawMoleculeCairo(smi, imageSizePx)
    if molImg is None:
        return None
    if not caption:
        return molImg
    banner = makeCaptionBanner(caption, molImg.width)
    panel  = Image.new("RGB", (molImg.width, molImg.height + banner.height), "white")
    panel.paste(molImg, (0, 0))
    panel.paste(banner, (0, molImg.height))
    return panel

def joinLabels(labelList):
    return " + ".join([l for l in labelList if l])

def putLabelCentered(draw, text, centerX, y, font, panelWidth, color="#555555"):
    tw = textWidth(draw, text, font)
    x  = int(centerX - tw / 2)
    x  = max(4, min(x, panelWidth - tw - 4))
    draw.text((x, y), text, fill=color, font=font)

def makeHorizontalArrow(width, height, consumedText, producedText, feasibilityText="", color=(30, 30, 30), lineWidth=8):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    cy    = height // 2
    left  = int(width * 0.10)
    right = int(width * 0.90)

    # main straight reaction arrow (left to right)
    draw.line([(left, cy), (right, cy)], fill=color, width=lineWidth)
    head = int(min(width, height) * 0.045)
    draw.polygon(
        [(right - int(head * 2.2), cy - head), (right - int(head * 2.2), cy + head), (right, cy)],
        fill=color,
    )

    # feasibility score, centered just below the main arrow, independent of cofactor labels
    if showFeasibilityScore and feasibilityText:
        scoreY = cy + int(0.12 * height)
        putLabelCentered(draw, feasibilityText, (left + right) // 2, scoreY, feasibilityFont, width, color="#1a1a1a")

    if not showCofactorLabels or (not consumedText and not producedText):
        return img

    # curved feeder: reactant enters at the left (tail) end, curve dips down to the
    # main arrow, product leaves at the right (arrowhead) end. Drawn as a quadratic
    # bezier so the endpoints sit under the two labels and the head follows the tangent.
    span  = right - left
    xL    = int(left + 0.15 * span)     # tail end, under the consumed label
    xR    = int(right - 0.05 * span)    # head end, under the produced label
    yTop  = cy - int(0.30 * height)     # both ends sit above the main arrow
    p0    = (xL, yTop)
    ctrl  = ((xL + xR) / 2.0, cy + 0.34 * height)   # control below pulls the curve down to the arrow
    p1    = (xR, yTop)

    thin = max(3, lineWidth // 2)
    pts  = []
    steps = 48
    for i in range(steps + 1):
        t  = i / steps
        mt = 1.0 - t
        x  = mt * mt * p0[0] + 2 * mt * t * ctrl[0] + t * t * p1[0]
        y  = mt * mt * p0[1] + 2 * mt * t * ctrl[1] + t * t * p1[1]
        pts.append((x, y))
    draw.line(pts, fill=color, width=thin, joint="curve")

    # arrowhead at the product (right) end, oriented along the curve tangent ctrl -> p1
    dx, dy = p1[0] - ctrl[0], p1[1] - ctrl[1]
    norm   = math.hypot(dx, dy) or 1.0
    ux, uy = dx / norm, dy / norm
    px, py = -uy, ux
    ah     = int(height * 0.05)
    tip    = p1
    base   = (p1[0] - ux * ah * 1.7, p1[1] - uy * ah * 1.7)
    draw.polygon([
        tip,
        (base[0] + px * ah, base[1] + py * ah),
        (base[0] - px * ah, base[1] - py * ah),
    ], fill=color)

    # labels above each end: consumed at the tail (left), produced at the head (right)
    labelY = yTop - 40
    if consumedText:
        putLabelCentered(draw, consumedText, xL, labelY, cofactorFont, width)
    if producedText:
        putLabelCentered(draw, producedText, xR, labelY, cofactorFont, width)
    return img

def makeVerticalArrow(width, height, consumedText, producedText, feasibilityText="", color=(30, 30, 30), lineWidth=8):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    cx     = width // 2
    top    = int(height * 0.14)
    bottom = int(height * 0.86)
    draw.line([(cx, top), (cx, bottom)], fill=color, width=lineWidth)
    head = int(min(width, height) * 0.10)
    draw.polygon(
        [(cx - head, bottom - int(head * 1.6)), (cx + head, bottom - int(head * 1.6)), (cx, bottom)],
        fill=color,
    )
    if showFeasibilityScore and feasibilityText:
        putLabelCentered(draw, feasibilityText, cx, bottom + 6, feasibilityFont, width, color="#1a1a1a")
    midY = (top + bottom) // 2
    if showCofactorLabels and consumedText:
        draw.text((cx + 18, midY - 40), consumedText, fill="#555555", font=cofactorFont)
    if showCofactorLabels and producedText:
        draw.text((cx + 18, midY + 8), producedText, fill="#555555", font=cofactorFont)
    return img

def stackPanelsHorizontally(parts, bgColor="white"):
    valid = [p for p in parts if p is not None]
    if not valid:
        return None
    w = sum(p.width for p in valid)
    h = max(p.height for p in valid)
    canvas = Image.new("RGB", (w, h), bgColor)
    x = 0
    for p in valid:
        canvas.paste(p, (x, 0))
        x += p.width
    return canvas

def stackPanelsVertically(parts, bgColor="white"):
    valid = [p for p in parts if p is not None]
    if not valid:
        return None
    w = max(p.width for p in valid)
    h = sum(p.height for p in valid)
    canvas = Image.new("RGB", (w, h), bgColor)
    y = 0
    for p in valid:
        canvas.paste(p, ((w - p.width) // 2, y))
        y += p.height
    return canvas

def buildCascadeImage(rowData, imageSizePx, arrowSpanPx, direction, ruleCofactorMap):
    canonStarterSet = {canonicalizeSmiles(s) for s in splitStarters(rowData.get("starter_Canonical_SMILES", ""))}
    canonTargetSmi  = canonicalizeSmiles(rowData.get("targetSMILES", ""))
    orderedSteps    = reorderStepDictsForward(rowData["stepDicts"], canonStarterSet)
    nodes, edges    = buildBackbone(orderedSteps, canonStarterSet, canonTargetSmi, ruleCofactorMap)
    if len(nodes) < 2:
        return None

    parts   = []
    lastIdx = len(nodes) - 1
    for idx, smi in enumerate(nodes):
        caption = ""
        if showStarterTargetTags and idx == 0:
            caption = "Starter"
        elif showStarterTargetTags and idx == lastIdx:
            caption = "Target"
        parts.append(makeMoleculePanel(smi, caption, imageSizePx))
        if idx != lastIdx:
            edge            = edges[idx]
            consumedText    = joinLabels(edge["consumed"])
            producedText    = joinLabels(edge["produced"])
            score           = edge.get("feasibilityScore")
            feasibilityText = f"{feasibilityScorePrefix}{score:.3f}" if score is not None else ""
            if direction == "horizontal":
                parts.append(makeHorizontalArrow(arrowSpanPx, imageSizePx[1], consumedText, producedText, feasibilityText))
            else:
                parts.append(makeVerticalArrow(imageSizePx[0], arrowSpanPx, consumedText, producedText, feasibilityText))

    return stackPanelsHorizontally(parts) if direction == "horizontal" else stackPanelsVertically(parts)

def saveImageHighRes(image, basePathNoExt, dpi):
    pngPath = basePathNoExt + ".png"
    pdfPath = basePathNoExt + ".pdf"
    image.save(pngPath, dpi=(dpi, dpi))
    image.save(pdfPath, "PDF", resolution=float(dpi))
    return pngPath, pdfPath

ruleCofactorMap = buildRuleCofactorMap(rulesetPath)
scoreByReaction = buildFeasibilityScoreMap(reactionDF_DORAXGB_highFeasibility)

if saveFigures:
    os.makedirs(articleFiguresDir, exist_ok=True)

# Build route-level records directly from the BFS output. feasibleRoutes is a list of
# (numSteps, targetCanon, starterCanon, steps) where steps is a forward-ordered list of
# "reactants >> products" strings. Each step's feasibilityScore_rule3 is looked up by a
# canonical reactionKey so ordering and spacing differences do not break the match.
routeRecords = []
for routeIndex, (numSteps, targetCanon, starterCanon, steps) in enumerate(feasibleRoutes):
    stepDicts = [
        {
            "reactionSmiles"   : str(rxnString),
            "ruleName"         : "",
            "feasibilityScore" : scoreByReaction.get(reactionKey(str(rxnString))),
        }
        for rxnString in steps
    ]
    routeRecords.append({
        "routeId"                  : f"bfsRoute_{routeIndex:04d}",
        "numSteps"                 : int(numSteps),
        "starter_Canonical_SMILES" : starterCanon,
        "targetSMILES"             : targetCanon,
        "stepDicts"                : stepDicts,
    })

routeLevelDF = (
    pd.DataFrame(
        routeRecords,
        columns=["routeId", "numSteps", "starter_Canonical_SMILES", "targetSMILES", "stepDicts"],
    )
    .sort_values(["numSteps", "routeId"])
    .reset_index(drop=True)
)

totalSteps        = sum(len(rec["stepDicts"]) for rec in routeRecords)
missingScoreSteps = sum(1 for rec in routeRecords for sd in rec["stepDicts"] if sd.get("feasibilityScore") is None)
if missingScoreSteps:
    print(f"Steps with no matched {feasibilityScoreColumn}: {missingScoreSteps:,} of {totalSteps:,}")

for numSteps, groupDF in routeLevelDF.groupby("numSteps"):
    availableRoutes = groupDF.reset_index(drop=True)
    indices         = selectedRoutes.get(numSteps, list(range(numToDisplay)))
    validIndices    = [i for i in indices if i < len(availableRoutes)]

    print(f"\n{numSteps}-step pathways: {len(availableRoutes):,} available, displaying {len(validIndices):,}")
    if not validIndices:
        print(f"  No valid indices for {numSteps}-step, check selectedRoutes config.")
        continue

    for rank, idx in enumerate(validIndices, start=1):
        rowData      = availableRoutes.loc[idx]
        cascadeImage = buildCascadeImage(rowData, moleculeImageSizePx, arrowSpanPx, layoutDirection, ruleCofactorMap)
        if cascadeImage is None:
            print(f"  [{rank}/{len(validIndices)}] Route: {rowData['routeId']} skipped, no drawable chain.")
            continue
        print(f"  [{rank}/{len(validIndices)}] Route: {rowData['routeId']}")
        display(cascadeImage)
        if saveFigures:
            basePath         = os.path.join(articleFiguresDir, f"{fileNamePrefix}_pathway_{rowData['routeId']}_{numSteps}step")
            pngPath, pdfPath = saveImageHighRes(cascadeImage, basePath, outputDpi)
            print(f"      saved: {pngPath}")
            print(f"      saved: {pdfPath}")

print("\nDone.")

## Plot pathways for all starter compounds

These renderers reproduce the original notebook and now run for **every**
starter. The helper cells below define the drawing routines once; the final
cell loops over all starters, and for each one shows the feasible pathways
from that starter to its high-feasibility target compounds, using that
starter's own network depth from its `doranet_config.yaml`.

### Pathway rendering helpers (vertical, per reaction step)

In [ ]:
# CONFIG
numToDisplayPerStep = 4
saveVerticalFigures         = False
figureOutDir        = os.path.join(resultsDir, "PathwayFigures")
subImgSize          = (600, 300)
articleFiguresDir     = os.path.join(resultsDir, "PathwayFigures")
rulesetPath           = rulesetPathForPlots
cofactorTablePath     = cofactorTablePathForPlots
gapSize = 14

cofactorDisplayOverride = {
    "CO-A": "CoA", "ACETYL-COA": "Acetyl-CoA", "OXYGEN-MOLECULE": "O2",
    "WATER": "H2O", "HYDROGEN-PEROXIDE": "H2O2", "CARBON-DIOXIDE": "CO2",
    "AMMONIA": "NH3", "S-ADENOSYLMETHIONINE": "SAM", "ADENOSYL-HOMO-CYS": "SAH",
    "2-KETOGLUTARATE": "2-Ketoglutarate", "GLT": "Glutamate", "PROTON": "H+",
}

def loadFont(size):
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

separatorFont = loadFont(52)
labelFont     = loadFont(18)
headerFont    = loadFont(22)

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

def cofactorKey(smi):
    mol = Chem.MolFromSmiles(str(smi), sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except Exception:
        return None

def loadCofactorNames(path):
    nameByKey = {}
    if not os.path.exists(path):
        print(f"Cofactor table not found at {path}; cofactor names will not be applied.")
        return nameByKey
    table = pd.read_csv(path, sep="\t")
    if "SMILES" not in table.columns or "Name" not in table.columns:
        print(f"Cofactor table {path} lacks Name/SMILES columns; names not applied.")
        return nameByKey
    for _, row in table.iterrows():
        key = cofactorKey(row["SMILES"])
        if key is None:
            continue
        rawName = str(row["Name"]).strip()
        nameByKey.setdefault(key, cofactorDisplayOverride.get(rawName, rawName))
    return nameByKey

cofactorNameByKey = loadCofactorNames(cofactorTablePath)

def cofactorNameFor(smi):
    return cofactorNameByKey.get(cofactorKey(smi))

def rdkitImageToPil(imageObj):
    if isinstance(imageObj, Image.Image):
        return imageObj.convert("RGB")
    if isinstance(imageObj, (bytes, bytearray)):
        return Image.open(BytesIO(imageObj)).convert("RGB")
    return Image.open(BytesIO(imageObj.data)).convert("RGB")

def centeredText(draw, text, imgWidth, y, font, color="black"):
    try:
        bbox  = draw.textbbox((0, 0), text, font=font)
        textW = bbox[2] - bbox[0]
    except Exception:
        textW = len(text) * 8
    draw.text(((imgWidth - textW) // 2, y), text, fill=color, font=font)

def makeTextBanner(text, bannerWidth, bannerHeight=40, bgColor="white", font=None):
    img = Image.new("RGB", (bannerWidth, bannerHeight), bgColor)
    centeredText(ImageDraw.Draw(img), text, bannerWidth, (bannerHeight - 24) // 2, font or headerFont)
    return img

def stackPanelsVertically(panelList, gapSize=14, bgColor="white"):
    valid = [p for p in panelList if p is not None]
    if not valid:
        return None
    w = max(p.width for p in valid)
    h = sum(p.height for p in valid) + gapSize * (len(valid) - 1)
    canvas = Image.new("RGB", (w, h), bgColor)
    y = 0
    for p in valid:
        canvas.paste(p, ((w - p.width) // 2, y))
        y += p.height + gapSize
    return canvas

def saveImageHighRes(image, basePathNoExt, dpi):
    pngPath = basePathNoExt + ".png"
    pdfPath = basePathNoExt + ".pdf"
    image.save(pngPath, dpi=(dpi, dpi))
    image.save(pdfPath, "PDF", resolution=float(dpi))
    return pngPath, pdfPath

def reorderStepsForward(steps, canonStarterSet):
    if len(steps) <= 1:
        return steps
    parsed = []
    for smi in steps:
        if ">>" not in str(smi):
            continue
        rStr, pStr = str(smi).split(">>", 1)
        parsed.append({
            "smi"      : smi,
            "reactants": {canonicalizeSmiles(s.strip()) for s in rStr.split(".") if s.strip()},
            "products" : {canonicalizeSmiles(s.strip()) for s in pStr.split(".") if s.strip()},
        })
    if not parsed:
        return steps
    firstIdx = next((i for i, s in enumerate(parsed) if s["reactants"] & canonStarterSet), None)
    if firstIdx is None:
        return steps
    ordered   = [parsed[firstIdx]]
    remaining = [s for i, s in enumerate(parsed) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["products"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactants"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return [s["smi"] for s in ordered]

def getMolLabel(smi, canonStarterSet, canonTargetSmi, side="reactant"):
    canon = canonicalizeSmiles(smi)
    if canon in canonStarterSet:
        return "Starter compound"
    if canon == canonTargetSmi:
        return "Target compound"
    cofactorName = cofactorNameFor(smi)
    if cofactorName:
        return cofactorName
    return "Reactant" if side == "reactant" else "Product"

def makeLabeledMolImage(smi, label, subImageSize=(280, 210)):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    molImg      = rdkitImageToPil(Draw.MolToImage(mol, size=subImageSize))
    labelBanner = Image.new("RGB", (molImg.width, 34), "#f0f0f0")
    centeredText(ImageDraw.Draw(labelBanner), label, molImg.width, 7, labelFont, "#333333")
    combined = Image.new("RGB", (molImg.width, molImg.height + labelBanner.height), "white")
    combined.paste(molImg,      (0, 0))
    combined.paste(labelBanner, (0, molImg.height))
    return combined

def makeSeparatorImage(text, height, width=90):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        bbox         = draw.textbbox((0, 0), text, font=separatorFont)
        textW, textH = bbox[2] - bbox[0], bbox[3] - bbox[1]
    except Exception:
        textW, textH = 30, 50
    draw.text(((width - textW) // 2, (height - textH) // 2), text, fill="#222222", font=separatorFont)
    return img

def makeCustomReactionPanel(rxnSmi, canonStarterSet, canonTargetSmi,
                             panelTitle, subImageSize=(280, 210)):
    if ">>" not in str(rxnSmi):
        return None
    reactantStr, productStr = str(rxnSmi).split(">>", 1)
    reactantSmiles = (
        [s for s in [x.strip() for x in reactantStr.split(".") if x.strip()] if canonicalizeSmiles(s) in canonStarterSet]
        + [s for s in [x.strip() for x in reactantStr.split(".") if x.strip()] if canonicalizeSmiles(s) not in canonStarterSet]
    )
    productSmiles = (
        [s for s in [x.strip() for x in productStr.split(".") if x.strip()] if canonicalizeSmiles(s) != canonTargetSmi]
        + [s for s in [x.strip() for x in productStr.split(".") if x.strip()] if canonicalizeSmiles(s) == canonTargetSmi]
    )
    reactantImgs = [img for img in [
        makeLabeledMolImage(s, getMolLabel(s, canonStarterSet, canonTargetSmi, side="reactant"), subImageSize)
        for s in reactantSmiles] if img is not None]
    productImgs  = [img for img in [
        makeLabeledMolImage(s, getMolLabel(s, canonStarterSet, canonTargetSmi, side="product"), subImageSize)
        for s in productSmiles]  if img is not None]
    if not reactantImgs or not productImgs:
        return None
    maxH  = max(img.height for img in reactantImgs + productImgs)
    gap   = 16
    parts = []
    for i, img in enumerate(reactantImgs):
        parts.append(img)
        if i < len(reactantImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=80))
    parts.append(makeSeparatorImage("→", maxH, width=100))
    for i, img in enumerate(productImgs):
        parts.append(img)
        if i < len(productImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=80))
    totalW = sum(p.width for p in parts) + gap * (len(parts) - 1)
    canvas = Image.new("RGB", (totalW, maxH), "white")
    x = 0
    for part in parts:
        canvas.paste(part, (x, (maxH - part.height) // 2))
        x += part.width + gap
    titleBanner = makeTextBanner(panelTitle, canvas.width, bannerHeight=38)
    final = Image.new("RGB", (canvas.width, canvas.height + titleBanner.height + 10), "white")
    final.paste(titleBanner, (0, 0))
    final.paste(canvas,      (0, titleBanner.height + 10))
    return final


### Pathway rendering helpers (horizontal cascade)

In [ ]:
# CONFIG
numToDisplay           = 5
selectedRoutes         = {}   # empty: every step-count group shows its first numToDisplay routes
layoutDirection        = "horizontal"
moleculeImageSizePx    = (900, 680)
arrowSpanPx            = 520
outputDpi              = 1200
showStarterTargetTags  = True
showCofactorLabels     = True
showFeasibilityScore   = True
feasibilityScoreColumn = feasibilityScoreCol   # column in reactionDF_DORAXGB_highFeasibility
feasibilityScorePrefix = ""                          # set to "feasibility = " to prefix the number
saveCascadeFigures            = True
maxCofactorHeavyAtoms  = 12   # unknown species larger than this are treated as co-products, not cofactors

# Which starters to plot. Give up to nStartersToPlot names explicitly, or leave the list
# empty to auto-pick the starters with the most high-feasibility candidates.
startersToPlot   = []   # e.g. ["Basidalin", "BasidalinPrecursor_starter_00007", ...]
nStartersToPlot  = 6

# Optional display-name overrides, keyed by the Name in all_cofactors.tsv.
# Leave empty to show the table names verbatim (CO-A, OXYGEN-MOLECULE, ...).
cofactorDisplayOverride = {
    "CO-A": "CoA",
    "ACETYL-COA": "Acetyl-CoA",
    "OXYGEN-MOLECULE": "O2",
    "WATER": "H2O",
    "HYDROGEN-PEROXIDE": "H2O2",
    "CARBON-DIOXIDE": "CO2",
    "AMMONIA": "NH3",
    "S-ADENOSYLMETHIONINE": "SAM",
    "ADENOSYL-HOMO-CYS": "SAH",
    "2-KETOGLUTARATE": "2-Ketoglutarate",
    "GLT": "Glutamate",
    "PROTON": "H+",
}

def loadFont(size):
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

captionFont     = loadFont(30)
cofactorFont    = loadFont(26)
feasibilityFont = loadFont(28)

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

def cofactorKey(smi):
    """
    Canonical, stereo- and charge-insensitive key. Tolerates '*' wildcard atoms
    in the cofactor table (FAD, F420, quinones, formyl-THF).
    """
    mol = Chem.MolFromSmiles(str(smi), sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except Exception:
        return None

def loadCofactorNames(path):
    """Map canonical SMILES key -> display name, sourced from all_cofactors.tsv."""
    nameByKey = {}
    if not os.path.exists(path):
        print(f"Cofactor table not found at {path}; cofactors will fall back to formulas.")
        return nameByKey
    table = pd.read_csv(path, sep="\t")
    if "SMILES" not in table.columns or "Name" not in table.columns:
        print(f"Cofactor table {path} lacks Name/SMILES columns; names not applied.")
        return nameByKey
    for _, row in table.iterrows():
        key = cofactorKey(row["SMILES"])
        if key is None:
            continue
        rawName = str(row["Name"]).strip()
        nameByKey.setdefault(key, cofactorDisplayOverride.get(rawName, rawName))
    return nameByKey

cofactorNameByKey = loadCofactorNames(cofactorTablePath)

def isKnownCofactor(smi):
    return cofactorKey(smi) in cofactorNameByKey

def labelSpecies(smi):
    key = cofactorKey(smi)
    if key in cofactorNameByKey:
        return cofactorNameByKey[key]
    mol = Chem.MolFromSmiles(str(smi))
    return rdMolDescriptors.CalcMolFormula(mol) if mol else str(smi)

# Rule-level role tokens, used only when a reaction carries no explicit cofactor molecule.
cofactorTokenNameMap = {
    "WATER": "H2O", "HYDROGEN_PEROXIDE": "H2O2", "OXYGEN": "O2", "O2": "O2",
    "CARBON_DIOXIDE": "CO2", "AMMONIA": "NH3", "AMMONIUM": "NH4+",
    "NAD": "NAD+", "NAD_PLUS": "NAD+", "NADH": "NADH",
    "NADP": "NADP+", "NADP_PLUS": "NADP+", "NADPH": "NADPH",
    "L_GLUTAMATE": "Glutamate", "GLUTAMATE": "Glutamate",
    "2_KETOGLUTARATE": "2-Ketoglutarate", "2_OXOGLUTARATE": "2-Oxoglutarate",
    "METHYL_DONOR_COF": "SAM", "METHYL_ACCEPTOR_COF": "SAH",
    "ATP": "ATP", "ADP": "ADP", "AMP": "AMP",
    "PHOSPHATE": "Pi", "DIPHOSPHATE": "PPi", "COA": "CoA", "ACETYL_COA": "Acetyl-CoA",
}
primaryRoleTokens = {"ANY", "ANY_COF", ""}

def heavyAtomCount(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return mol.GetNumHeavyAtoms() if mol else 0

def splitStarters(starterStr):
    return [] if pd.isna(starterStr) else [x.strip() for x in str(starterStr).split(";") if x.strip()]

def dedupeByCanon(smilesList):
    seen, kept = set(), []
    for s in smilesList:
        if not s:
            continue
        c = canonicalizeSmiles(s)
        if c not in seen:
            seen.add(c)
            kept.append(s)
    return kept

def dedupePreserveOrder(items):
    seen, kept = set(), []
    for x in items:
        if x and x not in seen:
            seen.add(x)
            kept.append(x)
    return kept

def cleanRoleToken(token):
    key = token.strip().upper().replace("-", "_").replace(" ", "_")
    if key in cofactorTokenNameMap:
        return cofactorTokenNameMap[key]
    return token.strip().replace("_", " ").title()

def extractRuleCofactors(cellValue):
    tokens = [t.strip() for t in re.split(r"[;,]", str(cellValue)) if t.strip()]
    labels = [cleanRoleToken(t) for t in tokens if t.strip().upper() not in primaryRoleTokens]
    return dedupePreserveOrder(labels)

def buildRuleCofactorMap(path):
    ruleMap = {}
    if not os.path.exists(path):
        print(f"  ruleset not found at {path}, using reaction-level cofactors only.")
        return ruleMap
    ruleset = pd.read_csv(path, sep="\t")
    nameCol = "Name" if "Name" in ruleset.columns else ruleset.columns[0]
    hasReact, hasProd = "Reactants" in ruleset.columns, "Products" in ruleset.columns
    for _, row in ruleset.iterrows():
        name = str(row.get(nameCol, "")).strip()
        if not name:
            continue
        entry = {
            "consumed": extractRuleCofactors(row["Reactants"]) if hasReact else [],
            "produced": extractRuleCofactors(row["Products"]) if hasProd else [],
        }
        ruleMap.setdefault(name, entry)
        ruleMap.setdefault(name.split("_")[0], entry)
    return ruleMap

def ruleCofactorsForStep(ruleName, ruleCofactorMap):
    name = str(ruleName).strip()
    entry = ruleCofactorMap.get(name) or ruleCofactorMap.get(name.split("_")[0])
    return (entry["consumed"], entry["produced"]) if entry else ([], [])

def parseReactionSides(reactionSmiles):
    if ">>" not in str(reactionSmiles):
        return [], []
    rStr, pStr = str(reactionSmiles).split(">>", 1)
    return (
        [s.strip() for s in rStr.split(".") if s.strip()],
        [s.strip() for s in pStr.split(".") if s.strip()],
    )

def reactionKey(reactionSmiles):
    """Order- and spacing-independent reaction identity: canonical reactants and products."""
    reactants, products = parseReactionSides(reactionSmiles)
    reactantKey = ".".join(sorted(canonicalizeSmiles(s) for s in reactants))
    productKey  = ".".join(sorted(canonicalizeSmiles(s) for s in products))
    return reactantKey + ">>" + productKey

def buildFeasibilityScoreMap(scoreDF):
    """Map canonical reactionKey -> feasibilityScore_rule3 from reactionDF_DORAXGB_highFeasibility."""
    if feasibilityScoreColumn not in scoreDF.columns:
        raise KeyError(f"Column '{feasibilityScoreColumn}' not in reactionDF_DORAXGB_highFeasibility; available: {list(scoreDF.columns)}")
    if "reactionString" in scoreDF.columns:
        reactionSeries = scoreDF["reactionString"].astype(str)
    elif "reactants" in scoreDF.columns and "products" in scoreDF.columns:
        reactionSeries = scoreDF["reactants"].astype(str) + " >> " + scoreDF["products"].astype(str)
    else:
        raise KeyError("reactionDF_DORAXGB_highFeasibility needs a 'reactionString' column or both 'reactants' and 'products' columns to key reactions.")
    scoreByReaction = {}
    for reactionStr, score in zip(reactionSeries, scoreDF[feasibilityScoreColumn]):
        try:
            scoreValue = float(score)
        except (TypeError, ValueError):
            continue
        scoreByReaction.setdefault(reactionKey(reactionStr), scoreValue)
    return scoreByReaction

def reorderStepDictsForward(stepDicts, canonStarterSet):
    if len(stepDicts) <= 1:
        return stepDicts
    enriched = []
    for sd in stepDicts:
        reactants, products = parseReactionSides(sd["reactionSmiles"])
        if not reactants and not products:
            continue
        enriched.append({
            **sd,
            "reactantSet": {canonicalizeSmiles(s) for s in reactants},
            "productSet" : {canonicalizeSmiles(s) for s in products},
        })
    if not enriched:
        return stepDicts
    firstIdx = next((i for i, s in enumerate(enriched) if s["reactantSet"] & canonStarterSet), None)
    if firstIdx is None:
        return stepDicts
    ordered   = [enriched[firstIdx]]
    remaining = [s for i, s in enumerate(enriched) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["productSet"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactantSet"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return ordered

def pickCarrier(products, prevCarrierCanon, canonTargetSmi, nextReactants):
    # never let a known cofactor become a backbone node
    pool = [p for p in products if canonicalizeSmiles(p) != prevCarrierCanon and not isKnownCofactor(p)]
    if not pool:
        pool = [p for p in products if canonicalizeSmiles(p) != prevCarrierCanon] or products
    carrier = next((p for p in pool if canonicalizeSmiles(p) == canonTargetSmi), None)
    if carrier is None and nextReactants is not None:
        nextCanon = {canonicalizeSmiles(s) for s in nextReactants}
        feeding = [p for p in pool if canonicalizeSmiles(p) in nextCanon]
        if feeding:
            carrier = max(feeding, key=heavyAtomCount)
    if carrier is None:
        carrier = max(pool, key=heavyAtomCount)
    return carrier

def resolveCofactorLabels(step, reactants, products, prevCarrierCanon, carrierCanon, ruleCofactorMap):
    def keepAsCofactor(smi):
        return isKnownCofactor(smi) or heavyAtomCount(smi) <= maxCofactorHeavyAtoms

    consumedSpecies = [s for s in dedupeByCanon([r for r in reactants if canonicalizeSmiles(r) != prevCarrierCanon]) if keepAsCofactor(s)]
    producedSpecies = [s for s in dedupeByCanon([p for p in products  if canonicalizeSmiles(p) != carrierCanon])   if keepAsCofactor(s)]
    consumedLabels = [labelSpecies(s) for s in consumedSpecies]
    producedLabels = [labelSpecies(s) for s in producedSpecies]

    ruleConsumed, ruleProduced = ruleCofactorsForStep(step.get("ruleName", ""), ruleCofactorMap)
    if not consumedLabels and not producedLabels and (ruleConsumed or ruleProduced):
        consumedLabels, producedLabels = ruleConsumed, ruleProduced

    return dedupePreserveOrder(consumedLabels), dedupePreserveOrder(producedLabels)

def buildBackbone(stepDicts, canonStarterSet, canonTargetSmi, ruleCofactorMap):
    steps = []
    for sd in stepDicts:
        reactants, products = parseReactionSides(sd["reactionSmiles"])
        if not reactants or not products:
            continue
        steps.append({**sd, "reactants": reactants, "products": products})
    if not steps:
        return [], []

    firstReactants = steps[0]["reactants"]
    startCarrier = next((s for s in firstReactants if canonicalizeSmiles(s) in canonStarterSet), None)
    if startCarrier is None:
        nonCofactorFirst = [s for s in firstReactants if not isKnownCofactor(s)] or firstReactants
        startCarrier = max(nonCofactorFirst, key=heavyAtomCount)
    nodes            = [startCarrier]
    edges            = []
    prevCarrierCanon = canonicalizeSmiles(startCarrier)

    for stepIdx, step in enumerate(steps):
        reactants, products = step["reactants"], step["products"]
        nextReactants = steps[stepIdx + 1]["reactants"] if stepIdx + 1 < len(steps) else None
        carrier      = pickCarrier(products, prevCarrierCanon, canonTargetSmi, nextReactants)
        carrierCanon = canonicalizeSmiles(carrier)
        consumedLabels, producedLabels = resolveCofactorLabels(
            step, reactants, products, prevCarrierCanon, carrierCanon, ruleCofactorMap
        )
        nodes.append(carrier)
        edges.append({
            "consumed"         : consumedLabels,
            "produced"         : producedLabels,
            "feasibilityScore" : step.get("feasibilityScore"),
        })
        prevCarrierCanon = carrierCanon
    return nodes, edges

def drawMoleculeCairo(smi, sizePx):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    AllChem.Compute2DCoords(mol)
    drawer = rdMolDraw2D.MolDraw2DCairo(sizePx[0], sizePx[1])
    opts = drawer.drawOptions()
    opts.bondLineWidth   = 3
    opts.padding         = 0.10
    opts.clearBackground = True
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    return Image.open(BytesIO(drawer.GetDrawingText())).convert("RGB")

def textWidth(draw, text, font):
    try:
        bbox = draw.textbbox((0, 0), text, font=font)
        return bbox[2] - bbox[0]
    except Exception:
        return len(text) * 12

def centeredTextX(draw, text, boxWidth, y, font, color="#333333"):
    draw.text(((boxWidth - textWidth(draw, text, font)) // 2, y), text, fill=color, font=font)

def makeCaptionBanner(text, width, height=46):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    centeredTextX(draw, text, width, (height - 34) // 2, captionFont)
    return img

def makeMoleculePanel(smi, caption, imageSizePx):
    molImg = drawMoleculeCairo(smi, imageSizePx)
    if molImg is None:
        return None
    if not caption:
        return molImg
    banner = makeCaptionBanner(caption, molImg.width)
    panel  = Image.new("RGB", (molImg.width, molImg.height + banner.height), "white")
    panel.paste(molImg, (0, 0))
    panel.paste(banner, (0, molImg.height))
    return panel

def joinLabels(labelList):
    return " + ".join([l for l in labelList if l])

def putLabelCentered(draw, text, centerX, y, font, panelWidth, color="#555555"):
    tw = textWidth(draw, text, font)
    x  = int(centerX - tw / 2)
    x  = max(4, min(x, panelWidth - tw - 4))
    draw.text((x, y), text, fill=color, font=font)

def makeHorizontalArrow(width, height, consumedText, producedText, feasibilityText="", color=(30, 30, 30), lineWidth=8):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    cy    = height // 2
    left  = int(width * 0.10)
    right = int(width * 0.90)

    # main straight reaction arrow (left to right)
    draw.line([(left, cy), (right, cy)], fill=color, width=lineWidth)
    head = int(min(width, height) * 0.045)
    draw.polygon(
        [(right - int(head * 2.2), cy - head), (right - int(head * 2.2), cy + head), (right, cy)],
        fill=color,
    )

    # feasibility score, centered just below the main arrow, independent of cofactor labels
    if showFeasibilityScore and feasibilityText:
        scoreY = cy + int(0.12 * height)
        putLabelCentered(draw, feasibilityText, (left + right) // 2, scoreY, feasibilityFont, width, color="#1a1a1a")

    if not showCofactorLabels or (not consumedText and not producedText):
        return img

    span  = right - left
    xL    = int(left + 0.15 * span)     # tail end, under the consumed label
    xR    = int(right - 0.05 * span)    # head end, under the produced label
    yTop  = cy - int(0.30 * height)     # both ends sit above the main arrow
    p0    = (xL, yTop)
    ctrl  = ((xL + xR) / 2.0, cy + 0.34 * height)   # control below pulls the curve down to the arrow
    p1    = (xR, yTop)

    thin = max(3, lineWidth // 2)
    pts  = []
    steps = 48
    for i in range(steps + 1):
        t  = i / steps
        mt = 1.0 - t
        x  = mt * mt * p0[0] + 2 * mt * t * ctrl[0] + t * t * p1[0]
        y  = mt * mt * p0[1] + 2 * mt * t * ctrl[1] + t * t * p1[1]
        pts.append((x, y))
    draw.line(pts, fill=color, width=thin, joint="curve")

    dx, dy = p1[0] - ctrl[0], p1[1] - ctrl[1]
    norm   = math.hypot(dx, dy) or 1.0
    ux, uy = dx / norm, dy / norm
    px, py = -uy, ux
    ah     = int(height * 0.05)
    tip    = p1
    base   = (p1[0] - ux * ah * 1.7, p1[1] - uy * ah * 1.7)
    draw.polygon([
        tip,
        (base[0] + px * ah, base[1] + py * ah),
        (base[0] - px * ah, base[1] - py * ah),
    ], fill=color)

    labelY = yTop - 40
    if consumedText:
        putLabelCentered(draw, consumedText, xL, labelY, cofactorFont, width)
    if producedText:
        putLabelCentered(draw, producedText, xR, labelY, cofactorFont, width)
    return img

def makeVerticalArrow(width, height, consumedText, producedText, feasibilityText="", color=(30, 30, 30), lineWidth=8):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    cx     = width // 2
    top    = int(height * 0.14)
    bottom = int(height * 0.86)
    draw.line([(cx, top), (cx, bottom)], fill=color, width=lineWidth)
    head = int(min(width, height) * 0.10)
    draw.polygon(
        [(cx - head, bottom - int(head * 1.6)), (cx + head, bottom - int(head * 1.6)), (cx, bottom)],
        fill=color,
    )
    if showFeasibilityScore and feasibilityText:
        putLabelCentered(draw, feasibilityText, cx, bottom + 6, feasibilityFont, width, color="#1a1a1a")
    midY = (top + bottom) // 2
    if showCofactorLabels and consumedText:
        draw.text((cx + 18, midY - 40), consumedText, fill="#555555", font=cofactorFont)
    if showCofactorLabels and producedText:
        draw.text((cx + 18, midY + 8), producedText, fill="#555555", font=cofactorFont)
    return img

def stackPanelsHorizontally(parts, bgColor="white"):
    valid = [p for p in parts if p is not None]
    if not valid:
        return None
    w = sum(p.width for p in valid)
    h = max(p.height for p in valid)
    canvas = Image.new("RGB", (w, h), bgColor)
    x = 0
    for p in valid:
        canvas.paste(p, (x, 0))
        x += p.width
    return canvas

def stackPanelsVertically(panelList, gapSize=14, bgColor="white"):
    valid = [p for p in panelList if p is not None]
    if not valid:
        return None
    w = max(p.width for p in valid)
    h = sum(p.height for p in valid) + gapSize * (len(valid) - 1)
    canvas = Image.new("RGB", (w, h), bgColor)
    y = 0
    for p in valid:
        canvas.paste(p, ((w - p.width) // 2, y))
        y += p.height + gapSize
    return canvas

def buildCascadeImage(rowData, imageSizePx, arrowSpanPx, direction, ruleCofactorMap):
    canonStarterSet = {canonicalizeSmiles(s) for s in splitStarters(rowData.get("starter_Canonical_SMILES", ""))}
    canonTargetSmi  = canonicalizeSmiles(rowData.get("targetSMILES", ""))
    orderedSteps    = reorderStepDictsForward(rowData["stepDicts"], canonStarterSet)
    nodes, edges    = buildBackbone(orderedSteps, canonStarterSet, canonTargetSmi, ruleCofactorMap)
    if len(nodes) < 2:
        return None

    parts   = []
    lastIdx = len(nodes) - 1
    for idx, smi in enumerate(nodes):
        caption = ""
        if showStarterTargetTags and idx == 0:
            caption = "Starter"
        elif showStarterTargetTags and idx == lastIdx:
            caption = "Target"
        parts.append(makeMoleculePanel(smi, caption, imageSizePx))
        if idx != lastIdx:
            edge            = edges[idx]
            consumedText    = joinLabels(edge["consumed"])
            producedText    = joinLabels(edge["produced"])
            score           = edge.get("feasibilityScore")
            feasibilityText = f"{feasibilityScorePrefix}{score:.3f}" if score is not None else ""
            if direction == "horizontal":
                parts.append(makeHorizontalArrow(arrowSpanPx, imageSizePx[1], consumedText, producedText, feasibilityText))
            else:
                parts.append(makeVerticalArrow(imageSizePx[0], arrowSpanPx, consumedText, producedText, feasibilityText))

    return stackPanelsHorizontally(parts) if direction == "horizontal" else stackPanelsVertically(parts)

def saveImageHighRes(image, basePathNoExt, dpi):
    pngPath = basePathNoExt + ".png"
    pdfPath = basePathNoExt + ".pdf"
    image.save(pngPath, dpi=(dpi, dpi))
    image.save(pdfPath, "PDF", resolution=float(dpi))
    return pngPath, pdfPath

ruleCofactorMap = buildRuleCofactorMap(rulesetPath)


def selectStartersToPlot(explicit, n):
    """Return the starter names to plot: the explicit list (kept only where high-feasibility
    data exists), or the n starters with the most high-feasibility candidates."""
    plottable = [
        name for name, ctx in starterContexts.items()
        if ctx.get("reactionDF_highFeasible") is not None
        and ctx.get("generatedCompoundsDF_highFeasibility") is not None
    ]
    if explicit:
        chosen = [s for s in explicit if s in plottable]
        missing = [s for s in explicit if s not in plottable]
        if missing:
            print(f"Skipping (not found or no high-feasibility data): {missing}")
        return chosen
    ranked = sorted(
        plottable,
        key=lambda name: len(starterContexts[name]["generatedCompoundsDF_highFeasibility"]),
        reverse=True,
    )
    return ranked[:n]

### Render pathways for every starter

In [ ]:
def plotPathwaysForStarter(starterName):
    ctx = starterContexts[starterName]
    if ctx["reactionDF_highFeasible"] is None:
        print(f"[skip] {starterName}: no high-feasibility data (missing {feasibilityFileName})")
        return

    # Per-starter variables the original drivers read.
    reactionDF_highFeasible = ctx["reactionDF_highFeasible"]
    reactionDF_DORAXGB_highFeasibility = ctx["reactionDF_highFeasible"]
    generatedCompoundsDF_highFeasibility = ctx["generatedCompoundsDF_highFeasibility"]
    canonStarterSet = ctx["canonStarterSet"]
    canonHelperSet = ctx["canonHelperSet"]
    doranet_generations = ctx["generations"]
    fileNamePrefix = starterName

    print(f"\n########## {starterName}  (network depth {doranet_generations}) ##########")

    # Feasible-reaction network (all steps here are DORA-XGB high-feasibility by construction)
    feasibleRxnDF = reactionDF_highFeasible.copy()
    if "reactionString" not in feasibleRxnDF.columns:
        feasibleRxnDF["reactionString"] = (
            feasibleRxnDF["reactants"].astype(str) + " >> " + feasibleRxnDF["products"].astype(str)
        )

    # canonical reactant -> feasible reactions consuming it
    reactantToFeasibleRxns = defaultdict(list)
    for idx, row in feasibleRxnDF.iterrows():
        for smi in splitMoleculeString(row["reactants"]):
            reactantToFeasibleRxns[canonicalizeSmiles(smi)].append(idx)

    def carrierProductsOf(row):
        """Non-cofactor, non-helper, non-starter products that can be transformed further."""
        out = []
        for productSmi in splitMoleculeString(row["products"]):
            productCanon = canonicalizeSmiles(productSmi)
            if productCanon in canonHelperSet:
                continue
            if productCanon in canonStarterSet:
                continue
            productMol = Chem.MolFromSmiles(str(productSmi))
            if isCofactorOrEndogenous(productMol, productCanon, cofactorExclusionSet, physicochemicalCutoffs):
                continue
            out.append(productCanon)
        return out

    # Depth-capped BFS over feasible edges: shortest fully-feasible route (<= doranet_generations) to each compound.
    # parent[compound] = (rxnIdx, precursorCanon); absence means "no all-feasible route within doranet_generations".
    parentReaction = {}
    seenCompounds  = set(canonStarterSet)
    searchQueue    = deque((c, 0) for c in canonStarterSet)

    while searchQueue:
        currentCanon, currentDepth = searchQueue.popleft()
        if currentDepth >= doranet_generations:
            continue
        for rxnIdx in reactantToFeasibleRxns.get(currentCanon, []):
            row = feasibleRxnDF.loc[rxnIdx]
            for productCanon in carrierProductsOf(row):
                if productCanon in seenCompounds:
                    continue
                seenCompounds.add(productCanon)
                parentReaction[productCanon] = (rxnIdx, currentCanon)
                searchQueue.append((productCanon, currentDepth + 1))

    def feasibleRouteTo(targetCanon):
        """Reaction strings of the fully-feasible route to a target, forward order; [] if none within doranet_generations."""
        steps = []
        cur   = targetCanon
        while cur in parentReaction:
            rxnIdx, precursorCanon = parentReaction[cur]
            steps.append(feasibleRxnDF.loc[rxnIdx, "reactionString"])
            cur = precursorCanon
        steps.reverse()
        return steps

    def reorderStepsForward(steps, canonStarterSet):
        if len(steps) <= 1:
            return steps
        parsed = []
        for smi in steps:
            if ">>" not in str(smi):
                continue
            rStr, pStr = str(smi).split(">>", 1)
            parsed.append({
                "smi"      : smi,
                "reactants": {canonicalizeSmiles(s.strip()) for s in rStr.split(".") if s.strip()},
                "products" : {canonicalizeSmiles(s.strip()) for s in pStr.split(".") if s.strip()},
            })
        if not parsed:
            return steps
        firstIdx = next((i for i, s in enumerate(parsed) if s["reactants"] & canonStarterSet), None)
        if firstIdx is None:
            return steps
        ordered   = [parsed[firstIdx]]
        remaining = [s for i, s in enumerate(parsed) if i != firstIdx]
        while remaining:
            prevProducts = ordered[-1]["products"]
            nextIdx = next((i for i, s in enumerate(remaining) if s["reactants"] & prevProducts), None)
            if nextIdx is None:
                ordered.extend(remaining)
                break
            ordered.append(remaining[nextIdx])
            remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
        return [s["smi"] for s in ordered]

    def buildRouteImageFromSteps(stepReactions, starterCanon, targetCanon,
                                 headerText, subImageSize, bannerWidth):
        canonStarterSetLocal = {starterCanon}
        numSteps             = len(stepReactions)
        forwardSteps         = reorderStepsForward(stepReactions, canonStarterSetLocal)
        panelBlocks = [makeTextBanner(headerText, bannerWidth=bannerWidth, bannerHeight=44, bgColor="#eef2f7")]
        for stepIdx, rxnSmi in enumerate(forwardSteps, start=1):
            panel = makeCustomReactionPanel(
                rxnSmi, canonStarterSetLocal, targetCanon,
                panelTitle=f"Reaction step {stepIdx} of {numSteps}",
                subImageSize=subImageSize,
            )
            if panel:
                panelBlocks.append(panel)
        return stackPanelsVertically(panelBlocks, gapSize=14, bgColor="white")

    if saveVerticalFigures:
        os.makedirs(figureOutDir, exist_ok=True)

    candidateDF = generatedCompoundsDF_highFeasibility.reset_index(drop=True)

    # Keep only candidates that have a fully-feasible route within doranet_generations
    feasibleRoutes = []
    for _, cand in candidateDF.iterrows():
        targetCanon = canonicalizeSmiles(cand["Canonical_SMILES"])
        steps       = feasibleRouteTo(targetCanon)
        if steps and len(steps) <= doranet_generations:
            firstReactants = str(steps[0]).split(">>", 1)[0]
            starterCanon   = next(
                (canonicalizeSmiles(s) for s in firstReactants.split(".") if canonicalizeSmiles(s) in canonStarterSet),
                cand.get("starter_Canonical_SMILES", ""),
            )
            feasibleRoutes.append((len(steps), targetCanon, starterCanon, steps))

    feasibleRoutes.sort(key=lambda t: t[0])

    print(f"High-feasibility candidates                 : {len(candidateDF):,}")
    print(f"With a fully-feasible route (<= {doranet_generations} steps) : {len(feasibleRoutes):,}")
    stepHistogram = defaultdict(int)
    for nSteps, *_ in feasibleRoutes:
        stepHistogram[nSteps] += 1
    for nSteps in sorted(stepHistogram):
        print(f"  {nSteps}-step feasible pathways : {stepHistogram[nSteps]:,}")

    drawnByStep = defaultdict(int)
    for numSteps, targetCanon, starterCanon, steps in feasibleRoutes:
        if drawnByStep[numSteps] >= numToDisplayPerStep:
            continue
        drawnByStep[numSteps] += 1
        headerText = f"Starter: {str(starterCanon)[:40]}  |  Target: {targetCanon[:50]}  |  Steps: {numSteps}"
        routeImage = buildRouteImageFromSteps(steps, starterCanon, targetCanon, headerText, subImgSize, bannerWidth=1800)
        if routeImage is None:
            continue
        print(f"  [{numSteps}-step #{drawnByStep[numSteps]}] {targetCanon}")
        display(routeImage)
        if saveVerticalFigures:
            basePath         = os.path.join(figureOutDir, f"highFeasPathway_{numSteps}step_{drawnByStep[numSteps]}")
            pngPath, pdfPath = saveImageHighRes(routeImage, basePath, outputDpi)
            print(f"      saved: {pngPath}")

    print("\nDone.")

    scoreByReaction = buildFeasibilityScoreMap(reactionDF_DORAXGB_highFeasibility)

    if saveCascadeFigures:
        os.makedirs(articleFiguresDir, exist_ok=True)

    # Build route-level records directly from the BFS output. feasibleRoutes is a list of
    # (numSteps, targetCanon, starterCanon, steps) where steps is a forward-ordered list of
    # "reactants >> products" strings. Each step's feasibilityScore_rule3 is looked up by a
    # canonical reactionKey so ordering and spacing differences do not break the match.
    routeRecords = []
    for routeIndex, (numSteps, targetCanon, starterCanon, steps) in enumerate(feasibleRoutes):
        stepDicts = [
            {
                "reactionSmiles"   : str(rxnString),
                "ruleName"         : "",
                "feasibilityScore" : scoreByReaction.get(reactionKey(str(rxnString))),
            }
            for rxnString in steps
        ]
        routeRecords.append({
            "routeId"                  : f"bfsRoute_{routeIndex:04d}",
            "numSteps"                 : int(numSteps),
            "starter_Canonical_SMILES" : starterCanon,
            "targetSMILES"             : targetCanon,
            "stepDicts"                : stepDicts,
        })

    routeLevelDF = (
        pd.DataFrame(
            routeRecords,
            columns=["routeId", "numSteps", "starter_Canonical_SMILES", "targetSMILES", "stepDicts"],
        )
        .sort_values(["numSteps", "routeId"])
        .reset_index(drop=True)
    )

    totalSteps        = sum(len(rec["stepDicts"]) for rec in routeRecords)
    missingScoreSteps = sum(1 for rec in routeRecords for sd in rec["stepDicts"] if sd.get("feasibilityScore") is None)
    if missingScoreSteps:
        print(f"Steps with no matched {feasibilityScoreColumn}: {missingScoreSteps:,} of {totalSteps:,}")

    for numSteps, groupDF in routeLevelDF.groupby("numSteps"):
        availableRoutes = groupDF.reset_index(drop=True)
        indices         = selectedRoutes.get(numSteps, list(range(numToDisplay)))
        validIndices    = [i for i in indices if i < len(availableRoutes)]

        print(f"\n{numSteps}-step pathways: {len(availableRoutes):,} available, displaying {len(validIndices):,}")
        if not validIndices:
            print(f"  No valid indices for {numSteps}-step, check selectedRoutes config.")
            continue

        for rank, idx in enumerate(validIndices, start=1):
            rowData      = availableRoutes.loc[idx]
            cascadeImage = buildCascadeImage(rowData, moleculeImageSizePx, arrowSpanPx, layoutDirection, ruleCofactorMap)
            if cascadeImage is None:
                print(f"  [{rank}/{len(validIndices)}] Route: {rowData['routeId']} skipped, no drawable chain.")
                continue
            print(f"  [{rank}/{len(validIndices)}] Route: {rowData['routeId']}")
            display(cascadeImage)
            if saveCascadeFigures:
                basePath         = os.path.join(articleFiguresDir, f"{fileNamePrefix}_pathway_{rowData['routeId']}_{numSteps}step")
                pngPath, pdfPath = saveImageHighRes(cascadeImage, basePath, outputDpi)
                print(f"      saved: {pngPath}")
                print(f"      saved: {pdfPath}")

    print("\nDone.")


# Render pathways for some starter that has high-feasibility data.
namesToPlot = selectStartersToPlot(startersToPlot, nStartersToPlot)
print(f"Plotting pathways for {len(namesToPlot)} starters: {namesToPlot}\n")
for starterName in namesToPlot:
    plotPathwaysForStarter(starterName)